In [5]:
import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.signal import savgol_filter
import os, warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS — matched to original MF-DFA code's conventions
# ─────────────────────────────────────────────────────────────────────────────
TICKER        = "CL=F"
START         = "2024-01-01"
END           = "2026-04-17"

Q_VALS        = np.arange(-30, 31, 5, dtype=float)
POLY_ORD      = 1
OVERLAP       = 0.75
N_SCALES      = 30
SCALE_MIN     = 20
SCALE_MAX     = 800

REGIME_WINDOW = 120
VAR_WINDOW    = 60
CONFIDENCE    = 0.99
POSITION_SIZE = 100_000

H_PERSIST     = 0.60
H_ANTIPERSIST = 0.45
DALPHA_BLOWOFF= 0.40

VAR_MULTIPLIERS = {
    "I":          1.0,
    "II":         1.6,
    "III":        2.2,
    "transition": 1.3,
}

OUT_DIR = "regime_var_plots"
os.makedirs(OUT_DIR, exist_ok=True)

REGIME_COLORS = {"I": "#27500A", "II": "#854F0B", "III": "#A32D2D", "transition": "#5F5E5A"}
REGIME_BG     = {"I": "#EAF3DE", "II": "#FAEEDA", "III": "#FCEBEB", "transition": "#F1EFE8"}

# Underestimation severity thresholds for annotation markers
UNDEREST_SEVERE_PCT = 30.0   # mark points where standard VaR underestimates by >30%

# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA
# ─────────────────────────────────────────────────────────────────────────────
def prepare_series(price):
    price     = price[price > 0]
    log_price = np.log(price)
    returns   = np.diff(log_price)
    return price, log_price, returns

def integrate(x):
    return np.cumsum(x - x.mean())

def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)

def mfdfa(x, q_vals, scale_min, scale_max, n_scales, poly_ord=1, overlap=0.75):
    y = integrate(x)
    N = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    if scale_max < scale_min:
        return np.array([]), np.full((len(q_vals), 0), np.nan), np.full(len(q_vals), np.nan)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    if len(scales) < 4:
        return scales, np.full((len(q_vals), len(scales)), np.nan), np.full(len(q_vals), np.nan)

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue
        t, rms = np.arange(s), []
        for st in starts:
            seg  = y[st:st+s]
            coef = np.polyfit(t, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t))**2))
            if r > 0:
                rms.append(r)
        rms = np.asarray(rms)
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms**2)))
            else:
                Fqs[j, i] = (np.mean(rms**q))**(1.0/q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq

def multifractal_spectrum(q_vals, hq):
    tau_q = q_vals * hq - 1.0
    wl    = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_s = savgol_filter(tau_q, wl, 2) if len(tau_q) >= 5 else tau_q
    alpha = np.gradient(tau_s, q_vals)
    D_a   = q_vals * alpha - tau_s
    return tau_q, alpha, D_a

# ─────────────────────────────────────────────────────────────────────────────
# DIAGNOSTIC
# ─────────────────────────────────────────────────────────────────────────────
def check_window_viability(window, scale_min=SCALE_MIN, scale_max=SCALE_MAX,
                           n_scales=N_SCALES, poly_ord=POLY_ORD):
    N          = window
    eff_smax   = min(scale_max, N // 4)
    eff_smin   = max(scale_min, poly_ord + 2)
    viable     = eff_smax >= eff_smin
    if viable:
        scales  = np.unique(
            np.logspace(np.log10(eff_smin), np.log10(eff_smax), n_scales).astype(int)
        )
        scales  = scales[scales >= eff_smin]
        n_valid = len(scales)
    else:
        n_valid = 0
    print(f"  window={window:4d} → N//4={N//4:3d} | eff scale range [{eff_smin}, {eff_smax}] "
          f"| {n_valid} distinct scales | viable={viable}")
    return viable

print("Scale viability check for candidate window sizes:")
for w in [30, 60, 80, 100, 120, 150, 200]:
    check_window_viability(w)

# ─────────────────────────────────────────────────────────────────────────────
# REGIME CLASSIFIER
# ─────────────────────────────────────────────────────────────────────────────
def classify_regime(hq, q_vals, alpha):
    q2_idx = np.argmin(np.abs(q_vals - 2.0))
    h2     = hq[q2_idx] if np.isfinite(hq[q2_idx]) else np.nan
    valid  = np.isfinite(alpha)
    dalpha = (alpha[valid].max() - alpha[valid].min()) if valid.sum() >= 2 else np.nan

    if np.isnan(h2) or np.isnan(dalpha):
        return "insufficient_data", h2, dalpha

    if   h2 > H_PERSIST and dalpha < DALPHA_BLOWOFF:  return "I",          h2, dalpha
    elif h2 > H_PERSIST and dalpha >= DALPHA_BLOWOFF: return "II",         h2, dalpha
    elif h2 < H_ANTIPERSIST:                           return "III",        h2, dalpha
    else:                                              return "transition", h2, dalpha

# ─────────────────────────────────────────────────────────────────────────────
# VaR
# ─────────────────────────────────────────────────────────────────────────────
def historical_var(returns, confidence=CONFIDENCE):
    return -np.percentile(returns, (1 - confidence) * 100)

def var_comparison(returns, regime, confidence=CONFIDENCE, position=POSITION_SIZE):
    base_var    = historical_var(returns, confidence)
    multiplier  = VAR_MULTIPLIERS.get(regime, 1.3)
    adj_var     = base_var * multiplier
    underest_pct= (adj_var - base_var) / adj_var * 100 if adj_var > 0 else 0.0
    underest_usd= (adj_var - base_var) * position
    return {
        "std_var_pct":   round(base_var * 100, 4),
        "adj_var_pct":   round(adj_var  * 100, 4),
        "multiplier":    multiplier,
        "underest_pct":  round(underest_pct, 2),
        "underest_usd":  round(underest_usd, 2),
        "std_var_usd":   round(base_var * position, 2),
        "adj_var_usd":   round(adj_var  * position, 2),
    }

# ─────────────────────────────────────────────────────────────────────────────
# ROLLING PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def run_rolling(close_prices, dates):
    _, _, returns = prepare_series(close_prices)
    N = len(returns)

    eff_smin = max(SCALE_MIN, POLY_ORD + 2)
    eff_smax = min(SCALE_MAX, REGIME_WINDOW // 4)

    print(f"\nRolling MF-DFA: window={REGIME_WINDOW}, "
          f"scale range [{eff_smin}, {eff_smax}], N_SCALES={N_SCALES}")
    print(f"{'Date':<12} {'Regime':<18} {'H(q=2)':>7} {'Δα':>7} "
          f"{'StdVaR%':>8} {'AdjVaR%':>8} {'Gap$':>10} {'Underest%':>10}")
    print("─" * 90)

    out = []
    start_idx = max(REGIME_WINDOW, VAR_WINDOW)

    for i in range(start_idx, N):
        seg_r   = returns[i - REGIME_WINDOW : i]
        var_seg = returns[i - VAR_WINDOW    : i]

        _, _, hq    = mfdfa(seg_r, Q_VALS, eff_smin, eff_smax, N_SCALES, POLY_ORD, OVERLAP)
        _, alpha, _ = multifractal_spectrum(Q_VALS, hq)
        regime, h2, da = classify_regime(hq, Q_VALS, alpha)

        vr = var_comparison(var_seg, regime)

        print(f"{str(dates[i+1])[:10]:<12} {regime:<18} "
              f"{h2 if not np.isnan(h2) else float('nan'):>7.3f} "
              f"{da if not np.isnan(da) else float('nan'):>7.3f} "
              f"{vr['std_var_pct']:>8.4f} {vr['adj_var_pct']:>8.4f} "
              f"{vr['underest_usd']:>10,.0f} {vr['underest_pct']:>9.1f}%")

        out.append({
            "date":       dates[i+1],
            "regime":     regime,
            "h2":         h2,
            "da":         da,
            "price":      close_prices[i+1],   # price aligned to result date
            "ret":        returns[i],           # return on that day
            **vr
        })

    return out

# ─────────────────────────────────────────────────────────────────────────────
# PLOT — 6 panels: price, returns, H(q=2), VaR comparison, underest%, dollar gap
# ─────────────────────────────────────────────────────────────────────────────
def plot_results(results):
    dates        = [r["date"]         for r in results]
    regimes      = [r["regime"]       for r in results]
    h2s          = [r["h2"]           for r in results]
    dalphas      = [r["da"]           for r in results]
    std_vars     = [r["std_var_pct"]  for r in results]
    adj_vars     = [r["adj_var_pct"]  for r in results]
    underest_pct = [r["underest_pct"] for r in results]
    underest_usd = [r["underest_usd"] for r in results]
    prices       = [r["price"]        for r in results]
    rets         = [r["ret"]          for r in results]

    xs          = np.arange(len(dates))
    bar_colors  = [REGIME_COLORS.get(r, "#888780") for r in regimes]

    # Identify severe underestimation points: where standard VaR misses >threshold%
    severe_mask = np.array(underest_pct) >= UNDEREST_SEVERE_PCT

    # ── figure layout: 6 rows ─────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 20))
    gs  = gridspec.GridSpec(
        6, 1, figure=fig,
        height_ratios=[2, 1.4, 1.2, 1.4, 1.2, 1.2],
        hspace=0.45
    )
    axes = [fig.add_subplot(gs[i]) for i in range(6)]

    fig.suptitle(
        f"Regime-aware VaR  ·  {TICKER}  ·  {CONFIDENCE*100:.0f}% confidence  "
        f"·  regime window={REGIME_WINDOW}d",
        fontsize=14, fontweight="bold", y=0.995
    )

    # ── shared regime background shading ─────────────────────────────────
    def shade(ax):
        for j, r in enumerate(regimes):
            ax.axvspan(j - 0.5, j + 0.5,
                       color=REGIME_BG.get(r, "#F1EFE8"), alpha=0.55, lw=0)

    # ─── PANEL 0 — Price with regime shading & risk-underestimation markers ──
    ax = axes[0]
    shade(ax)
    ax.plot(xs, prices, color="#1A1A2E", lw=1.5, zorder=3, label="Price")

    # Scatter dots where severe underestimation occurs
    if severe_mask.any():
        ax.scatter(
            xs[severe_mask], np.array(prices)[severe_mask],
            s=55, color="#A32D2D", zorder=5, marker="v",
            label=f"Risk underestimated >{UNDEREST_SEVERE_PCT:.0f}% (std VaR)",
            edgecolors="#6B0000", linewidths=0.6
        )

    ax.set_ylabel("Price (USD)", fontsize=9)
    ax.set_title("(a) Price  ·  ▼ marks where standard VaR underestimates hidden risk",
                 fontsize=10, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper left")
    ax.tick_params(labelbottom=False)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda v, _: f"${v:,.0f}"))

    # ─── PANEL 1 — Daily returns with regime shading & underestimation markers ──
    ax = axes[1]
    shade(ax)

    # Colour each return bar by regime
    for j, (ret, regime) in enumerate(zip(rets, regimes)):
        ax.bar(j, ret * 100,
               color=REGIME_COLORS.get(regime, "#888780"),
               alpha=0.75, width=0.85)

    # Overlay triangles for severe underestimation days
    if severe_mask.any():
        ax.scatter(
            xs[severe_mask],
            np.array(rets)[severe_mask] * 100,
            s=55, color="#A32D2D", zorder=5, marker="v",
            edgecolors="#6B0000", linewidths=0.6,
            label=f"Risk underestimated >{UNDEREST_SEVERE_PCT:.0f}%"
        )

    ax.axhline(0, color="#888780", lw=0.6)
    ax.set_ylabel("Return (%)", fontsize=9)
    ax.set_title("(b) Daily returns  ·  coloured by regime  ·  ▼ = hidden risk zone",
                 fontsize=10, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper left")
    ax.tick_params(labelbottom=False)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda v, _: f"{v:.2f}%"))

    # ─── PANEL 2 — H(q=2) ────────────────────────────────────────────────
    ax = axes[2]
    shade(ax)
    ax.plot(xs, h2s, color="#185FA5", lw=1.4, label="H(q=2)")
    ax.axhline(H_PERSIST,     color="#27500A", lw=0.8, ls="--", alpha=0.7,
               label=f"Persist. threshold  H={H_PERSIST}")
    ax.axhline(H_ANTIPERSIST, color="#A32D2D", lw=0.8, ls="--", alpha=0.7,
               label=f"Anti-persist. threshold  H={H_ANTIPERSIST}")
    ax.set_ylabel("H(q=2)", fontsize=9)
    ax.set_title("(c) Generalised Hurst exponent H(q=2)", fontsize=10, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper right")
    ax.tick_params(labelbottom=False)

    # ─── PANEL 3 — VaR comparison ────────────────────────────────────────
    ax = axes[3]
    shade(ax)
    ax.plot(xs, std_vars, color="#888780", lw=1.2, ls="--", label="Standard VaR")
    ax.plot(xs, adj_vars, color="#A32D2D", lw=1.8, label="Regime-adjusted VaR")
    ax.fill_between(xs, std_vars, adj_vars,
                    color="#F09595", alpha=0.4, label="Hidden risk gap")
    # Mark severe underestimation episodes on this panel too
    if severe_mask.any():
        ax.scatter(xs[severe_mask], np.array(adj_vars)[severe_mask],
                   s=40, color="#A32D2D", zorder=5, marker="v",
                   edgecolors="#6B0000", linewidths=0.5)
    ax.set_ylabel("VaR (%)", fontsize=9)
    ax.set_title("(d) VaR comparison  ·  shaded gap = hidden risk",
                 fontsize=10, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper right")
    ax.tick_params(labelbottom=False)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda v, _: f"{v:.3f}%"))

    # ─── PANEL 4 — Underestimation % ─────────────────────────────────────
    ax = axes[4]
    shade(ax)
    ax.bar(xs, underest_pct, color=bar_colors, width=0.9, alpha=0.85)
    ax.axhline(UNDEREST_SEVERE_PCT, color="#A32D2D", lw=0.9, ls=":",
               label=f"Severe threshold ({UNDEREST_SEVERE_PCT:.0f}%)")
    ax.axhline(0, color="#888780", lw=0.6)
    ax.set_ylabel("Underest. (% of true risk)", fontsize=9)
    ax.set_title("(e) Fraction of true risk invisible to standard VaR",
                 fontsize=10, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper right")
    ax.tick_params(labelbottom=False)

    # ─── PANEL 5 — Dollar gap ─────────────────────────────────────────────
    ax = axes[5]
    shade(ax)
    ax.bar(xs, [u / 1000 for u in underest_usd],
           color=bar_colors, width=0.9, alpha=0.85)
    ax.set_ylabel("Hidden risk ($K)", fontsize=9)
    ax.set_title(f"(f) Dollar underestimation per ${POSITION_SIZE:,} position",
                 fontsize=10, fontweight="semibold")

    # ── x-axis ticks on last panel only ──────────────────────────────────
    xtick_step = max(1, len(xs) // 12)
    ax.set_xticks(xs[::xtick_step])
    ax.set_xticklabels(
        [str(dates[i])[:10] for i in xs[::xtick_step]],
        rotation=30, ha="right", fontsize=8
    )

    # ── shared legend: regime phases ─────────────────────────────────────
    regime_patches = [
        mpatches.Patch(color=REGIME_BG.get(r, "#F1EFE8"),
                       edgecolor=REGIME_COLORS.get(r, "#888780"), linewidth=0.8,
                       label=f"Phase {r}  ({REGIME_COLORS.get(r,'#888780')})")
        for r in ["I", "II", "III", "transition"]
    ]
    underest_marker = Line2D(
        [0], [0], marker="v", color="w", markerfacecolor="#A32D2D",
        markersize=8, markeredgecolor="#6B0000",
        label=f"Risk underestimated >{UNDEREST_SEVERE_PCT:.0f}% (▼)"
    )
    fig.legend(
        handles=regime_patches + [underest_marker],
        loc="lower center", ncol=5,
        fontsize=8.5, frameon=True, framealpha=0.9,
        bbox_to_anchor=(0.5, -0.015)
    )

    fname = f"{OUT_DIR}/{TICKER.replace('=','_')}_regime_var.png"
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\nSaved → {fname}")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import matplotlib.ticker   # needed inside FuncFormatter lambdas

    print(f"\nDownloading {TICKER}  {START} → {END}")
    data  = yf.download(TICKER, start=START, end=END, progress=False)
    close = data["Close"].dropna().values.ravel().astype(float)
    dates = data["Close"].dropna().index.tolist()
    print(f"  {len(close)} bars downloaded")

    results = run_rolling(close, dates)

    if results:
        plot_results(results)
    else:
        print("No results — extend START date or reduce REGIME_WINDOW.")

    print("\nDone.")

Scale viability check for candidate window sizes:
  window=  30 → N//4=  7 | eff scale range [20, 7] | 0 distinct scales | viable=False
  window=  60 → N//4= 15 | eff scale range [20, 15] | 0 distinct scales | viable=False
  window=  80 → N//4= 20 | eff scale range [20, 20] | 1 distinct scales | viable=True
  window= 100 → N//4= 25 | eff scale range [20, 25] | 6 distinct scales | viable=True
  window= 120 → N//4= 30 | eff scale range [20, 30] | 10 distinct scales | viable=True
  window= 150 → N//4= 37 | eff scale range [20, 37] | 18 distinct scales | viable=True
  window= 200 → N//4= 50 | eff scale range [20, 50] | 27 distinct scales | viable=True

  576 bars downloaded

Rolling MF-DFA: window=120, scale range [20, 30], N_SCALES=30
Date         Regime              H(q=2)      Δα  StdVaR%  AdjVaR%       Gap$  Underest%
──────────────────────────────────────────────────────────────────────────────────────────
2024-06-26   III                  0.311   1.058   3.6509   8.0321      4,381   

In [7]:
"""
Rényi-entropy MF-DFA volatility framework
==========================================
Replaces hardcoded VaR multipliers with a fully data-driven scaling factor
derived from the Rényi scaling exponent spectrum ψ(q).

Core idea
---------
For each scale s, define the normalised fluctuation measure over segments:

    p_i(s)  =  rms_i^2 / Σ_j rms_j^2          (probability-like weight)

Rényi entropy of order q:

    S_q(s)  =  1/(1-q) · log( Σ_i p_i(s)^q )   q ≠ 1
    S_1(s)  =  -Σ_i p_i(s) · log p_i(s)         (Shannon, limit q→1)

Scaling law:  S_q(s) ~ ψ(q) · log(s)

ψ(q) is the Rényi scaling exponent.  For a monofractal, ψ(q) is linear in q.
Curvature of ψ(q)  →  multifractality.

Volatility index (no free constants):
    residuals(q)  =  τ(q)  −  linear_fit(τ)(q)    [curvature of τ]
    V             =  std(residuals)                  [always ≥ 0]

    V = 0 for a monofractal (τ linear in q), V > 0 for multifractal.
    VaR multiplier: mult = 1 + V  — no lookup table, no regime boxes.

Connection to Pearson-correlation results
-----------------------------------------
The correlation r(q) between F_{+q} and F_{-q} drops away from 1 when the
fluctuation function is asymmetric across q.  Assets with low min-r (KO, PEP,
JNJ, CVX, MA, CL=F …) are exactly those where ψ(q) is most curved and V is
largest — i.e. where standard VaR is most dangerous to use uncorrected.

We flag each asset's "asymmetry severity" from the pre-computed min-r values
and show it alongside the Rényi-derived multiplier.
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy.signal import savgol_filter
import os, warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
ASSETS = {
    # ticker             : (sector label,  min_r from your correlation run)
    "KO"     : ("Healthcare & Consumer",  0.2383),
    "PEP"    : ("Healthcare & Consumer", -0.1332),
    "JNJ"    : ("Healthcare & Consumer",  0.3982),
    "CVX"    : ("Energy & Industrials",   0.5253),
    "MA"     : ("Financials",             0.7516),
    "CL=F"   : ("Commodities & Crypto",   0.7178),
    # add a 'stable' asset for contrast
    "AAPL"   : ("Tech",                   0.9700),
    "JPM"    : ("Financials",             0.9842),
}

START          = "2010-01-01"
END            = "2026-04-11"

# MF-DFA grid
Q_RENYI        = np.concatenate([
    np.linspace(-20, -0.5, 20),
    np.linspace( 0.5, 20, 20)
])                                # avoid q=0 (Shannon handled separately)
Q_LO_PCT       = 10               # percentile for q_lo
Q_HI_PCT       = 90               # percentile for q_hi

POLY_ORD       = 1
OVERLAP        = 0.75
N_SCALES       = 30
SCALE_MIN      = 20
SCALE_MAX      = 800

# Rolling VaR
REGIME_WINDOW  = 120
VAR_WINDOW     = 60
CONFIDENCE     = 0.99
POSITION_SIZE  = 100_000

OUT_DIR = "renyi_var_plots"
os.makedirs(OUT_DIR, exist_ok=True)

PALETTE = [
    "#f85149", "#58a6ff", "#3fb950", "#e3b341",
    "#bc8cff", "#ff7f0e", "#17becf", "#f78fb3",
]

# ─────────────────────────────────────────────────────────────────────────────
# CORE: segment RMS fluctuations (shared by both MF-DFA and Rényi)
# ─────────────────────────────────────────────────────────────────────────────
def integrate(x):
    return np.cumsum(x - x.mean())


def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


def segment_rms(x, scale_min, scale_max, n_scales,
                poly_ord=1, overlap=0.75):
    """
    Return (scales, rms_per_scale) where rms_per_scale[i] is the array of
    per-segment RMS values at scales[i].
    """
    y         = integrate(x)
    N         = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    if scale_max < scale_min:
        return np.array([]), []

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    rms_per_scale = []
    for s in scales:
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            rms_per_scale.append(np.array([]))
            continue
        t   = np.arange(s)
        rms = []
        for st in starts:
            seg  = y[st: st + s]
            coef = np.polyfit(t, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t)) ** 2))
            if r > 0:
                rms.append(r)
        rms_per_scale.append(np.asarray(rms))

    return scales, rms_per_scale


# ─────────────────────────────────────────────────────────────────────────────
# PARTITION FUNCTION  χ_q(s)  and Rényi scaling exponent  ψ(q)
# ─────────────────────────────────────────────────────────────────────────────
#
# Standard multifractal partition function (Chhabra & Jensen 1989):
#
#   p_i(s)   =  rms_i² / Σ_j rms_j²        normalised measure on segments
#   χ_q(s)   =  Σ_i p_i(s)^q               partition function of order q
#
# Scaling law:   log χ_q(s)  ~  τ(q) · log(s)
#
# Rényi entropy of order q:
#   S_q(s)  =  log χ_q(s) / (1 - q)        q ≠ 1
#   S_1(s)  =  -Σ_i p_i · log p_i          (Shannon, q→1 limit)
#
# Because  S_q(s) = log χ_q(s) / (1-q),  the scaling exponent of S_q is:
#   ψ(q) = τ(q) / (1 - q)
#
# But fitting log χ_q(s) directly gives τ(q) with large dynamic range.
# We use τ(q) as the primary quantity and derive ψ(q) = τ(q)/(1-q) for q≠1.
#
# Volatility index (no free constants):
#   Δτ  =  max_q τ(q)  −  min_q τ(q)       raw spread across q-orders
#   V   =  Δτ / Δq                          normalised by q-range
#
# For a monofractal: τ(q) = q·H − 1  (linear) → Δτ proportional to Δq → V≈H
# For a multifractal: τ(q) curves → Δτ exceeds linear baseline → V>H
# The *excess* curvature drives the multiplier: mult = 1 + max(curve, 0)

def _partition_function(rms, q):
    """
    log χ_q(s) = log Σ_i p_i^q   for a single scale.
    p_i = rms_i² / Σ rms_j²
    Returns NaN if undefined.
    """
    if len(rms) < 3:
        return np.nan
    p = rms ** 2 / (rms ** 2).sum()
    if q == 1:
        # Shannon entropy  S_1 = -Σ p log p  (serves as log χ proxy at q=1)
        p_safe = p[p > 0]
        return float(-np.sum(p_safe * np.log(p_safe)))
    val = np.sum(p ** q)
    if val <= 0:
        return np.nan
    return float(np.log(val))               # log χ_q(s)


def renyi_scaling_exponent(x, q_vals, scale_min, scale_max, n_scales,
                           poly_ord=1, overlap=0.75):
    """
    Fit  log χ_q(s)  vs  log(s)  for each q → slope is τ(q).
    Also return  ψ(q) = τ(q) / (1 - q)  (Rényi scaling exponent).

    Returns (q_vals, tau_q, psi_q).
    """
    scales, rms_per_scale = segment_rms(
        x, scale_min, scale_max, n_scales, poly_ord, overlap
    )
    if len(scales) == 0:
        nan = np.full(len(q_vals), np.nan)
        return q_vals, nan, nan

    n_scales_actual = len(scales)
    tau_q = np.full(len(q_vals), np.nan)
    psi_q = np.full(len(q_vals), np.nan)

    for j, q in enumerate(q_vals):
        log_chi = np.array([_partition_function(r, q) for r in rms_per_scale])
        mask    = np.isfinite(log_chi) & (np.array([len(r) for r in rms_per_scale]) >= 3)
        if mask.sum() < 4:
            continue
        # τ(q) = slope of log χ_q vs log s
        tau_q[j] = np.polyfit(np.log(scales[mask]), log_chi[mask], 1)[0]
        if abs(q - 1.0) > 1e-6:
            psi_q[j] = tau_q[j] / (1.0 - q)   # Rényi scaling exponent

    return q_vals, tau_q, psi_q


def renyi_volatility_index(q_vals, tau_q):
    """
    Data-derived volatility index from τ(q) curvature.

    For a monofractal:   τ(q) = q·H − 1  →  linear, zero curvature.
    For a multifractal:  τ(q) curves downward (concave).

    We measure curvature as the deviation from the best-fit linear τ:

        residuals(q)  =  τ(q)  −  linear_fit(τ)(q)
        V             =  std(residuals)             [always ≥ 0]

    This is purely data-driven: no thresholds, no lookup table.
    Typical range: 0 (monofractal) to ~0.5+ (strongly multifractal).
    VaR multiplier: mult = 1 + V.
    """
    valid = np.isfinite(tau_q)
    if valid.sum() < 6:
        return np.nan

    q_v   = q_vals[valid]
    tau_v = tau_q[valid]

    # fit and remove linear component of τ(q)
    lin_coef  = np.polyfit(q_v, tau_v, 1)
    tau_linear = np.polyval(lin_coef, q_v)
    residuals  = tau_v - tau_linear          # curvature signal

    return float(np.std(residuals))          # std of curvature → V


# ─────────────────────────────────────────────────────────────────────────────
# VaR (historical, no constants)
# ─────────────────────────────────────────────────────────────────────────────
def historical_var(returns, confidence=CONFIDENCE):
    return -np.percentile(returns, (1.0 - confidence) * 100.0)


def renyi_adjusted_var(returns_var_window, returns_regime_window,
                       confidence=CONFIDENCE, position=POSITION_SIZE):
    """
    Compute VaR multiplier entirely from ψ(q) spread — no lookup table.
    """
    s_min = max(SCALE_MIN, POLY_ORD + 2)
    s_max = min(SCALE_MAX, len(returns_regime_window) // 4)

    _, tau_q, psi_q = renyi_scaling_exponent(
        returns_regime_window, Q_RENYI, s_min, s_max, N_SCALES,
        poly_ord=POLY_ORD, overlap=OVERLAP
    )

    V          = renyi_volatility_index(Q_RENYI, tau_q)
    V          = V if np.isfinite(V) else 0.0   # std is always >= 0, no floor needed
    multiplier = 1.0 + V                         # data-derived, no constants

    base_var   = historical_var(returns_var_window, confidence)
    adj_var    = base_var * multiplier

    underest_pct = (adj_var - base_var) / adj_var * 100 if adj_var > 0 else 0.0
    underest_usd = (adj_var - base_var) * position

    return {
        "V":             round(V, 6),
        "multiplier":    round(multiplier, 4),
        "psi_q":         psi_q,
        "base_var_pct":  round(base_var * 100, 4),
        "adj_var_pct":   round(adj_var  * 100, 4),
        "underest_pct":  round(underest_pct, 2),
        "underest_usd":  round(underest_usd, 2),
    }


# ─────────────────────────────────────────────────────────────────────────────
# ROLLING PIPELINE FOR ONE ASSET
# ─────────────────────────────────────────────────────────────────────────────
def run_asset(ticker, close, dates):
    returns   = np.diff(np.log(close[close > 0]))
    N         = len(returns)
    start_idx = max(REGIME_WINDOW, VAR_WINDOW)

    rows = []
    for i in range(start_idx, N):
        seg_regime = returns[i - REGIME_WINDOW : i]
        seg_var    = returns[i - VAR_WINDOW    : i]

        res = renyi_adjusted_var(seg_var, seg_regime)
        rows.append({
            "date":  dates[i + 1],
            "price": close[i + 1],
            "ret":   returns[i],
            **{k: v for k, v in res.items() if k != "psi_q"},
        })

    return rows


# ─────────────────────────────────────────────────────────────────────────────
# FULL SPECTRUM ψ(q) ON ENTIRE SERIES  (static, for shape analysis)
# ─────────────────────────────────────────────────────────────────────────────
def full_series_psi(returns, ticker):
    s_min = max(SCALE_MIN, POLY_ORD + 2)
    s_max = min(SCALE_MAX, len(returns) // 4)
    _, tau_q, psi_q = renyi_scaling_exponent(
        returns, Q_RENYI, s_min, s_max, N_SCALES,
        poly_ord=POLY_ORD, overlap=OVERLAP
    )
    return tau_q, psi_q


# ─────────────────────────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
def plot_psi_spectra(psi_dict, min_r_dict):
    """Panel: ψ(q) for all assets — curvature reveals multifractality."""
    fig, ax = plt.subplots(figsize=(10, 5))

    for idx, (ticker, psi) in enumerate(psi_dict.items()):
        color   = PALETTE[idx % len(PALETTE)]
        min_r   = min_r_dict[ticker]
        valid   = np.isfinite(psi)
        label   = f"{ticker}  (min r={min_r:.3f})"
        lw      = 2.2 if min_r < 0.80 else 1.2
        ls      = "-" if min_r < 0.80 else "--"
        ax.plot(Q_RENYI[valid], psi[valid], color=color,
                lw=lw, ls=ls, label=label)

    ax.axhline(0, color="#555", lw=0.6, ls=":")
    ax.set_xlabel("|q|  (Rényi order)", fontsize=11)
    ax.set_ylabel("ψ(q)  [Rényi scaling exponent]", fontsize=11)
    ax.set_title(
        "ψ(q) spectrum — curvature = multifractality\n"
        "Solid lines: assets with strong correlation drop (low min-r)",
        fontsize=11
    )
    ax.legend(fontsize=8, ncol=2, loc="upper right", framealpha=0.9)
    ax.grid(alpha=0.2)

    fname = f"{OUT_DIR}/renyi_psi_spectra.png"
    fig.tight_layout()
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {fname}")


def plot_asset_rolling(ticker, rows, min_r, color):
    """4-panel rolling plot for a single asset."""
    dates        = [r["date"]         for r in rows]
    prices       = np.array([r["price"]        for r in rows])
    rets         = np.array([r["ret"]          for r in rows]) * 100
    Vs           = np.array([r["V"]            for r in rows])
    mults        = np.array([r["multiplier"]   for r in rows])
    base_vars    = np.array([r["base_var_pct"] for r in rows])
    adj_vars     = np.array([r["adj_var_pct"]  for r in rows])
    underest_pct = np.array([r["underest_pct"] for r in rows])
    underest_usd = np.array([r["underest_usd"] for r in rows])

    xs = np.arange(len(dates))

    # flag where multiplier is in the top quartile (most at risk)
    mult_thresh  = np.percentile(mults[np.isfinite(mults)], 75)
    severe_mask  = mults >= mult_thresh

    fig = plt.figure(figsize=(15, 14))
    gs  = gridspec.GridSpec(4, 1, figure=fig,
                            height_ratios=[1.8, 1.2, 1.3, 1.3], hspace=0.45)
    axes = [fig.add_subplot(gs[i]) for i in range(4)]

    fig.suptitle(
        f"Rényi-entropy VaR  ·  {ticker}  ·  {CONFIDENCE*100:.0f}% conf  "
        f"·  min Pearson r = {min_r:.3f}",
        fontsize=13, fontweight="bold"
    )

    # continuous V background (colour intensity = volatility index)
    def shade_by_V(ax):
        vmin, vmax = np.nanmin(Vs), np.nanmax(Vs)
        if vmax == vmin:
            return
        cmap = plt.cm.YlOrRd
        for j in range(len(xs)):
            intensity = (Vs[j] - vmin) / (vmax - vmin) if np.isfinite(Vs[j]) else 0
            ax.axvspan(j - 0.5, j + 0.5,
                       color=cmap(0.15 + 0.75 * intensity), alpha=0.35, lw=0)

    # ── panel 0: price ────────────────────────────────────────────────────
    ax = axes[0]
    shade_by_V(ax)
    ax.plot(xs, prices, color="#1A1A2E", lw=1.4, zorder=3)
    if severe_mask.any():
        ax.scatter(xs[severe_mask], prices[severe_mask],
                   s=55, color="#A32D2D", zorder=5, marker="v",
                   edgecolors="#6B0000", lw=0.6,
                   label=f"Top-quartile Rényi risk (mult ≥ {mult_thresh:.2f}×)")
    ax.set_ylabel("Price (USD)", fontsize=9)
    ax.set_title(f"(a) {ticker} price  ·  background intensity = Rényi volatility index V",
                 fontsize=9, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper left")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.tick_params(labelbottom=False)

    # ── panel 1: multiplier (data-derived, no constants) ─────────────────
    ax = axes[1]
    shade_by_V(ax)
    ax.plot(xs, mults, color=color, lw=1.6, label="mult = 1 + V(t)")
    ax.axhline(1.0, color="#555", lw=0.7, ls="--", alpha=0.6, label="No adjustment (mult=1)")
    ax.fill_between(xs, 1.0, mults, where=mults >= 1.0,
                    color=color, alpha=0.18, label="Excess risk load")
    ax.set_ylabel("VaR multiplier", fontsize=9)
    ax.set_title("(b) Data-derived multiplier  1 + V(t)  — no hardcoded constants",
                 fontsize=9, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper left")
    ax.tick_params(labelbottom=False)

    # ── panel 2: VaR comparison ───────────────────────────────────────────
    ax = axes[2]
    shade_by_V(ax)
    ax.plot(xs, base_vars, color="#888780", lw=1.2, ls="--", label="Standard VaR")
    ax.plot(xs, adj_vars,  color="#A32D2D", lw=1.8,           label="Rényi-adjusted VaR")
    ax.fill_between(xs, base_vars, adj_vars,
                    color="#F09595", alpha=0.4, label="Hidden risk gap")
    ax.set_ylabel("VaR (%)", fontsize=9)
    ax.set_title("(c) VaR — shaded gap is directly proportional to ψ(q) curvature",
                 fontsize=9, fontweight="semibold")
    ax.legend(fontsize=8, loc="upper right")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.3f}%"))
    ax.tick_params(labelbottom=False)

    # ── panel 3: dollar underestimation ──────────────────────────────────
    ax = axes[3]
    shade_by_V(ax)
    ax.bar(xs, underest_usd / 1000,
           color=[color if not s else "#A32D2D" for s in severe_mask],
           width=0.9, alpha=0.85)
    ax.set_ylabel("Hidden risk ($K)", fontsize=9)
    ax.set_title(f"(d) Dollar gap per ${POSITION_SIZE:,} — red bars = top-quartile Rényi risk",
                 fontsize=9, fontweight="semibold")

    xtick_step = max(1, len(xs) // 10)
    ax.set_xticks(xs[::xtick_step])
    ax.set_xticklabels([str(dates[i])[:10] for i in xs[::xtick_step]],
                       rotation=30, ha="right", fontsize=8)

    # colorbar for V intensity
    sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd,
                                norm=plt.Normalize(vmin=np.nanmin(Vs),
                                                   vmax=np.nanmax(Vs)))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation="vertical",
                        fraction=0.012, pad=0.01)
    cbar.set_label("Rényi volatility index V(t)", fontsize=9)

    fname = f"{OUT_DIR}/{ticker.replace('=','_').replace('-','_')}_renyi_var.png"
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {fname}")


def plot_multiplier_comparison(all_rows):
    """Cross-asset: rolling multiplier for every asset on one chart."""
    fig, ax = plt.subplots(figsize=(14, 5))

    for idx, (ticker, rows) in enumerate(all_rows.items()):
        color = PALETTE[idx % len(PALETTE)]
        xs    = np.arange(len(rows))
        mults = np.array([r["multiplier"] for r in rows])
        min_r = ASSETS[ticker][1]
        ax.plot(xs, mults, color=color, lw=1.5,
                label=f"{ticker}  (min r={min_r:.3f})")

    ax.axhline(1.0, color="#555", lw=0.7, ls="--", alpha=0.5, label="Baseline (1×)")
    ax.set_xlabel("Rolling window index", fontsize=10)
    ax.set_ylabel("Rényi multiplier  1 + V(t)", fontsize=10)
    ax.set_title(
        "Data-derived VaR multipliers across all assets\n"
        "Higher = more multifractal complexity = larger hidden risk",
        fontsize=11
    )
    ax.legend(fontsize=8, ncol=2, loc="upper right", framealpha=0.9)
    ax.grid(alpha=0.18)

    fname = f"{OUT_DIR}/renyi_multiplier_comparison.png"
    fig.tight_layout()
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    all_rows = {}
    psi_dict = {}

    for idx, (ticker, (sector, min_r)) in enumerate(ASSETS.items()):
        color = PALETTE[idx % len(PALETTE)]
        print(f"\n{'─'*60}")
        print(f"  {ticker}  [{sector}]  min Pearson r = {min_r:.4f}")
        print(f"{'─'*60}")

        data  = yf.download(ticker, start=START, end=END, progress=False)
        close = data["Close"].dropna().values.ravel().astype(float)
        dates = data["Close"].dropna().index.tolist()
        print(f"  {len(close)} bars downloaded")

        returns = np.diff(np.log(close[close > 0]))

        # full-series τ(q) and ψ(q) for spectrum plot
        print("  Computing full-series τ(q) / ψ(q) …")
        tau_full, psi_full = full_series_psi(returns, ticker)
        psi_dict[ticker]   = psi_full
        V_full = renyi_volatility_index(Q_RENYI, tau_full)
        print(f"  Full-series V = {V_full:.5f}   "
              f"→ implied multiplier = {1.0 + (V_full if np.isfinite(V_full) else 0.0):.4f}×")

        # rolling
        print("  Running rolling Rényi VaR …")
        rows = run_asset(ticker, close, dates)

        mults = [r["multiplier"] for r in rows]
        print(f"  Multiplier range: {min(mults):.3f}× – {max(mults):.3f}×  "
              f"(mean {np.mean(mults):.3f}×)")

        all_rows[ticker] = rows
        plot_asset_rolling(ticker, rows, min_r, color)

    # cross-asset ψ(q) spectra
    print("\nPlotting ψ(q) spectra …")
    plot_psi_spectra(psi_dict, {t: ASSETS[t][1] for t in ASSETS})

    # cross-asset multiplier comparison
    print("Plotting multiplier comparison …")
    plot_multiplier_comparison(all_rows)

    print("\nDone.  All plots saved to", OUT_DIR)


────────────────────────────────────────────────────────────
  KO  [Healthcare & Consumer]  min Pearson r = 0.2383
────────────────────────────────────────────────────────────
  4092 bars downloaded
  Computing full-series τ(q) / ψ(q) …
  Full-series V = 4.68112   → implied multiplier = 5.6811×
  Running rolling Rényi VaR …
  Multiplier range: 1.087× – 17.208×  (mean 4.341×)
  Saved → renyi_var_plots/KO_renyi_var.png

────────────────────────────────────────────────────────────
  PEP  [Healthcare & Consumer]  min Pearson r = -0.1332
────────────────────────────────────────────────────────────
  4092 bars downloaded
  Computing full-series τ(q) / ψ(q) …
  Full-series V = 4.41451   → implied multiplier = 5.4145×
  Running rolling Rényi VaR …
  Multiplier range: 1.158× – 15.252×  (mean 4.612×)
  Saved → renyi_var_plots/PEP_renyi_var.png

────────────────────────────────────────────────────────────
  JNJ  [Healthcare & Consumer]  min Pearson r = 0.3982
────────────────────────────────────

In [5]:
"""
MF-GARCH Conditional VaR Framework  —  optimised build
========================================================
Same model as v1, ~35x faster through three changes:

  Speed-up 1 — Vectorised MF-DFA inner loop
  -------------------------------------------
  Original: Python for-loop over every segment, np.polyfit per segment.
  Fixed:    Stack all segments as rows, solve one batched lstsq per scale.
    X  = [t | 1]         shape (s, 2)
    S  = row-stack segs  shape (n_segs, s)
    coef = lstsq(X, S.T) shape (2, n_segs)   — one LAPACK call
    resid = S - (X@coef).T                   — pure numpy broadcast
  5–6x faster per mfdfa_hq call.

  Speed-up 2 — GARCH: warm-start + periodic refit
  -------------------------------------------------
  Original: full L-BFGS-B from scratch every step (~38 ms, ~164 iters).
  Fixed:
    • Warm-start: pass previous solution as p0 → ~12 ms.
    • Refit only every GARCH_REFIT steps (default 5).
      Between refits, just re-run the cheap variance filter with cached
      params (0.08 ms). GARCH parameters barely move day-to-day.
  ~34x faster per GARCH call (amortised).

  Speed-up 3 — MF-DFA: periodic recompute
  -----------------------------------------
  MF-DFA uses a 500-day window. A 1-day slide shifts 0.2% of the data.
  The singularity spectrum barely changes. Recompute every MF_REFIT
  steps (default 5), cache Δα in between.

Combined: ~35x wall-clock speedup (27 min → ~45 sec for all 8 assets).

Model (unchanged from v1):
  VaR_t = -(μ + z_CF · σ_GARCH_t) · (1 + λ · clip(Δα_z, 0, 3))
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy import stats
from scipy.signal import savgol_filter
from scipy.optimize import minimize
import os, warnings, time

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
ASSETS = {
    "KO"   : ("Healthcare & Consumer",  0.2383),
    "PEP"  : ("Healthcare & Consumer", -0.1332),
    "JNJ"  : ("Healthcare & Consumer",  0.3982),
    "CVX"  : ("Energy & Industrials",   0.5253),
    "MA"   : ("Financials",             0.7516),
    "CL=F" : ("Commodities & Crypto",   0.7178),
    "AAPL" : ("Tech",                   0.9700),
    "JPM"  : ("Financials",             0.9842),
}

START          = "2010-01-01"
END            = "2026-04-11"

GARCH_WINDOW   = 250
MF_WINDOW      = 500
CF_WINDOW      = 120
DA_HISTORY     = 250

# Refit frequencies (key speed levers)
GARCH_REFIT    = 5     # refit GARCH every N steps; filter runs every step
MF_REFIT       = 5     # recompute MF-DFA every N steps

Q_VALS    = np.concatenate([np.arange(-10, 0, 1), np.arange(1, 11, 1)]).astype(float)
SCALE_MIN = 16
SCALE_MAX = 250
N_SCALES  = 20
POLY_ORD  = 1
OVERLAP   = 0.5

MF_LAMBDA  = 0.10
MF_Z_CAP   = 3.0
CONFIDENCE = 0.99
POSITION   = 100_000

OUT_DIR = "mf_garch_plots"
os.makedirs(OUT_DIR, exist_ok=True)

COLOR_STD     = "#888780"
COLOR_GARCH   = "#58a6ff"
COLOR_GARCHCF = "#e3b341"
COLOR_MFGARCH = "#f85149"
PALETTE = ["#f85149","#58a6ff","#3fb950","#e3b341",
           "#bc8cff","#ff7f0e","#17becf","#f78fb3"]

# Pre-build the detrending design matrix projection for each scale
# (computed once at import time, reused in every mfdfa call)
_PROJ_CACHE: dict = {}

def _get_proj(s: int):
    """
    Return (P, Qt) for scale s where P = X(X'X)^{-1}X' is the hat matrix.
    We store X and (X'X)^{-1}X' so residuals = seg - X @ ((X'X)^{-1}X' @ seg).
    Actually we just cache X and use lstsq — cache avoids rebuilding X each call.
    """
    if s not in _PROJ_CACHE:
        t = np.arange(s, dtype=np.float64)
        X = np.column_stack([t, np.ones(s)])
        _PROJ_CACHE[s] = X
    return _PROJ_CACHE[s]


# ─────────────────────────────────────────────────────────────────────────────
# LAYER 2 — VECTORISED MF-DFA
# ─────────────────────────────────────────────────────────────────────────────
def mfdfa_hq_fast(x, q_vals, scale_min, scale_max, n_scales,
                  poly_ord=1, overlap=0.5):
    """
    Vectorised MF-DFA: one batched lstsq per scale instead of one polyfit
    per segment.  Typically 5–6x faster than the loop version.
    """
    y         = np.cumsum(x - x.mean())
    N         = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)
    if scale_max < scale_min:
        return np.full(len(q_vals), np.nan)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs    = np.full((len(q_vals), len(scales)), np.nan)
    log_s  = np.log(scales.astype(float))

    for i, s in enumerate(scales):
        step   = max(1, int(s * (1.0 - overlap)))
        starts = np.arange(0, N - s + 1, step)
        if len(starts) < 3:
            continue

        X    = _get_proj(s)                                      # (s, 2)
        segs = np.stack([y[st: st + s] for st in starts])        # (n_seg, s)

        # Batch detrend: solve X @ C = segs.T  →  C shape (2, n_seg)
        C, _, _, _ = np.linalg.lstsq(X, segs.T, rcond=None)
        resid = segs - (X @ C).T                                  # (n_seg, s)
        rms   = np.sqrt(np.mean(resid ** 2, axis=1))              # (n_seg,)
        rms   = rms[rms > 0]
        if len(rms) == 0:
            continue

        rms2 = rms ** 2
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms2)))
            else:
                v = np.mean(rms ** q)
                if v > 0:
                    Fqs[j, i] = v ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(log_s[mask], np.log(row[mask]), 1)[0]
    return hq


def delta_alpha(hq, q_vals):
    """Δα = α_max - α_min from Legendre transform of τ(q) = q·H(q) - 1."""
    valid = np.isfinite(hq)
    if valid.sum() < 6:
        return np.nan
    q_v  = q_vals[valid]
    tau  = q_v * hq[valid] - 1.0
    wl   = min(5, len(tau) if len(tau) % 2 == 1 else len(tau) - 1)
    tau_s = savgol_filter(tau, window_length=max(wl, 3), polyorder=2) if len(tau) >= 5 else tau
    alpha = np.gradient(tau_s, q_v)
    fin   = np.isfinite(alpha)
    return float(alpha[fin].max() - alpha[fin].min()) if fin.sum() >= 2 else np.nan


# ─────────────────────────────────────────────────────────────────────────────
# LAYER 1 — GARCH(1,1)  with warm-start and periodic refit
# ─────────────────────────────────────────────────────────────────────────────
_GARCH_BOUNDS = [(1e-8, None), (1e-6, 0.5), (1e-6, 0.9999)]


def _garch_filter(params, returns):
    """Run the variance recursion. Pure Python but only 0.08 ms for 250 obs."""
    omega, alpha, beta = params
    n  = len(returns)
    h  = np.empty(n)
    h[0] = max(np.var(returns), 1e-10)
    for t in range(1, n):
        h[t] = omega + alpha * returns[t-1] ** 2 + beta * h[t-1]
    return np.maximum(h, 1e-10)


def _garch_negll(params, returns):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1.0:
        return 1e10
    h = _garch_filter(params, returns)
    return 0.5 * float(np.sum(np.log(h) + returns ** 2 / h))


def garch_fit_warm(returns, p0=None):
    """
    Fit GARCH with warm start from p0.  Tighter tolerance than v1
    (ftol=1e-7 instead of 1e-9) — saves ~30% iterations with negligible
    accuracy loss for rolling VaR purposes.
    """
    if p0 is None:
        v = np.var(returns)
        p0 = np.array([v * 0.05, 0.10, 0.85])

    result = minimize(_garch_negll, p0, args=(returns,),
                      method="L-BFGS-B", bounds=_GARCH_BOUNDS,
                      options={"maxiter": 200, "ftol": 1e-7})

    if result.success and result.x[1] + result.x[2] < 1.0:
        params = result.x
    else:
        # RiskMetrics fallback
        v = np.var(returns)
        params = np.array([v * 0.01, 0.06, 0.94])

    h        = _garch_filter(params, returns)
    sigma_t1 = float(np.sqrt(params[0]
                              + params[1] * returns[-1] ** 2
                              + params[2] * h[-1]))
    return params, np.sqrt(h), sigma_t1


# ─────────────────────────────────────────────────────────────────────────────
# LAYER 3 — CORNISH-FISHER on GARCH residuals
# ─────────────────────────────────────────────────────────────────────────────
def cf_zscore(residuals, confidence):
    g1   = float(stats.skew(residuals))
    g2   = float(stats.kurtosis(residuals))
    z    = stats.norm.ppf(1.0 - confidence)
    z_cf = (z
            + (z**2 - 1) * g1 / 6
            + (z**3 - 3*z) * g2 / 24
            - (2*z**3 - 5*z) * g1**2 / 36)
    return float(z_cf)


def historical_var(returns, confidence=CONFIDENCE):
    return float(-np.percentile(returns, (1.0 - confidence) * 100.0))


# ─────────────────────────────────────────────────────────────────────────────
# KUPIEC POF TEST
# ─────────────────────────────────────────────────────────────────────────────
def kupiec_pof(n_breaches, n_obs, confidence=CONFIDENCE):
    p0 = 1.0 - confidence
    if n_breaches == 0 or n_breaches >= n_obs:
        return np.nan, np.nan
    p_hat = n_breaches / n_obs
    try:
        lr   = -2.0 * (n_breaches * np.log(p0 / p_hat)
                       + (n_obs - n_breaches) * np.log((1 - p0) / (1 - p_hat)))
        return float(lr), float(1.0 - stats.chi2.cdf(lr, df=1))
    except Exception:
        return np.nan, np.nan


# ─────────────────────────────────────────────────────────────────────────────
# ROLLING PIPELINE  (optimised)
# ─────────────────────────────────────────────────────────────────────────────
def run_asset(ticker, close, dates):
    returns   = np.diff(np.log(close[close > 0]))
    N         = len(returns)
    start_idx = max(GARCH_WINDOW, MF_WINDOW, CF_WINDOW) + DA_HISTORY

    # ── Pre-compute Δα history for normalisation ──────────────────────────
    print(f"  Pre-computing Δα history (every {MF_REFIT} steps) …")
    da_raw = []
    for i in range(MF_WINDOW, min(start_idx + 1, N - 1), MF_REFIT):
        seg   = returns[i - MF_WINDOW: i]
        s_max = min(SCALE_MAX, len(seg) // 4)
        hq    = mfdfa_hq_fast(seg, Q_VALS, SCALE_MIN, s_max, N_SCALES,
                               POLY_ORD, OVERLAP)
        da_raw.append(delta_alpha(hq, Q_VALS))
    da_raw = np.array(da_raw)
    da_med_global = np.nanmedian(da_raw)
    da_std_global = max(np.nanstd(da_raw), 1e-6)
    print(f"  Δα history: n={len(da_raw)}, "
          f"mean={np.nanmean(da_raw):.4f}, std={da_std_global:.4f}")

    # ── Main rolling loop ─────────────────────────────────────────────────
    rows           = []
    garch_params   = None       # warm-start cache
    cached_da      = np.nan     # last Δα
    cached_da_z    = 0.0
    cached_mult    = 1.0
    z_base         = stats.norm.ppf(1.0 - CONFIDENCE)

    t_loop = time.perf_counter()
    for i in range(start_idx, N - 1):
        seg_g  = returns[i - GARCH_WINDOW: i]
        seg_mf = returns[i - MF_WINDOW:    i]

        # ── GARCH: refit every GARCH_REFIT steps, filter every step ──────
        if (i - start_idx) % GARCH_REFIT == 0:
            garch_params, sigma_hist, sigma_t1 = garch_fit_warm(seg_g, garch_params)
        else:
            sigma_hist = _garch_filter(garch_params, seg_g)
            sigma_t1   = float(np.sqrt(garch_params[0]
                                        + garch_params[1] * seg_g[-1] ** 2
                                        + garch_params[2] * sigma_hist[-1]))

        mu = float(np.mean(seg_g))

        # ── CF on GARCH residuals ─────────────────────────────────────────
        seg_cf = seg_g[-CF_WINDOW:]
        sig_cf = sigma_hist[-CF_WINDOW:]
        resids = (seg_cf - mu) / np.maximum(sig_cf, 1e-8)
        z_cf   = cf_zscore(resids, CONFIDENCE) if len(resids) >= 20 else z_base

        # ── MF-DFA: recompute every MF_REFIT steps ────────────────────────
        if (i - start_idx) % MF_REFIT == 0:
            s_max  = min(SCALE_MAX, len(seg_mf) // 4)
            hq     = mfdfa_hq_fast(seg_mf, Q_VALS, SCALE_MIN, s_max, N_SCALES,
                                    POLY_ORD, OVERLAP)
            da_val = delta_alpha(hq, Q_VALS)
            cached_da = da_val

            # rolling normalisation
            hist_end   = i - start_idx
            hist_start = max(0, hist_end - DA_HISTORY)
            slice_idx  = slice(hist_start // MF_REFIT, hist_end // MF_REFIT + 1)
            h_slice    = da_raw[slice_idx]
            da_med     = np.nanmedian(h_slice) if np.isfinite(h_slice).any() else da_med_global
            da_std     = max(np.nanstd(h_slice), 1e-6) if np.isfinite(h_slice).any() else da_std_global

            da_z          = np.clip((da_val - da_med) / da_std, 0.0, MF_Z_CAP) \
                            if np.isfinite(da_val) else 0.0
            cached_da_z   = da_z
            cached_mult   = 1.0 + MF_LAMBDA * da_z

        # ── Compose VaR ───────────────────────────────────────────────────
        var_std      = max(historical_var(seg_g[-60:], CONFIDENCE), 1e-6)
        var_garch    = max(-(mu + z_base * sigma_t1),   1e-6)
        var_garch_cf = max(-(mu + z_cf   * sigma_t1),   1e-6)
        var_mfgarch  = max(var_garch_cf * cached_mult,  1e-6)

        actual_loss = -returns[i]

        rows.append({
            "date"        : dates[i + 1],
            "price"       : close[i + 1],
            "ret"         : returns[i],
            "actual_loss" : actual_loss,
            "sigma_t1"    : sigma_t1,
            "da"          : cached_da,
            "da_z"        : cached_da_z,
            "mult_mf"     : cached_mult,
            "z_cf"        : z_cf,
            "var_std"     : var_std,
            "var_garch"   : var_garch,
            "var_garch_cf": var_garch_cf,
            "var_mfgarch" : var_mfgarch,
            "breach_std"      : actual_loss > var_std,
            "breach_garch"    : actual_loss > var_garch,
            "breach_garch_cf" : actual_loss > var_garch_cf,
            "breach_mfgarch"  : actual_loss > var_mfgarch,
        })

    elapsed = time.perf_counter() - t_loop
    print(f"  Loop: {len(rows)} steps in {elapsed:.1f}s "
          f"({elapsed/len(rows)*1000:.2f} ms/step)")
    return rows


# ─────────────────────────────────────────────────────────────────────────────
# VALIDATION SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
def validation_summary(rows, ticker, min_r):
    actual = np.array([r["actual_loss"] for r in rows])
    models = {
        "Standard"   : np.array([r["var_std"]        for r in rows]),
        "GARCH"      : np.array([r["var_garch"]       for r in rows]),
        "GARCH+CF"   : np.array([r["var_garch_cf"]    for r in rows]),
        "MF-GARCH+CF": np.array([r["var_mfgarch"]     for r in rows]),
    }
    results = {}
    for name, var_arr in models.items():
        n        = len(actual)
        breaches = actual > var_arr
        n_b      = breaches.sum()
        mask     = np.isfinite(var_arr)
        r2       = float(np.corrcoef(var_arr[mask], actual[mask])[0, 1] ** 2)
        excess   = actual[breaches] - var_arr[breaches]
        sev      = float(excess.mean()) if len(excess) > 0 else 0.0
        lr, pval = kupiec_pof(n_b, n)
        results[name] = dict(n_breach=n_b, br_pct=n_b/n*100,
                              r2=r2, sev=sev, kupiec_p=pval)

    print(f"\n  Validation — {ticker} (min r={min_r:.4f})  n={len(rows)}")
    print(f"  {'Model':<15} {'Breach%':>7} {'Target':>7} {'R²':>7} "
          f"{'Sev%':>8} {'Kupiec-p':>9}")
    print(f"  {'─'*57}")
    for name, v in results.items():
        print(f"  {name:<15} {v['br_pct']:>6.2f}% {'1.00%':>7} "
              f"{v['r2']:>7.4f} {v['sev']*100:>7.3f}% {v['kupiec_p']:>9.4f}")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
def plot_asset(ticker, rows, min_r, val_results):
    dates     = [r["date"]          for r in rows]
    prices    = np.array([r["price"]        for r in rows])
    actual    = np.array([r["actual_loss"]  for r in rows]) * 100
    sigma     = np.array([r["sigma_t1"]     for r in rows]) * 100
    da_arr    = np.array([r["da"]           for r in rows])
    mult_arr  = np.array([r["mult_mf"]      for r in rows])
    var_std   = np.array([r["var_std"]      for r in rows]) * 100
    var_garch = np.array([r["var_garch"]    for r in rows]) * 100
    var_gcf   = np.array([r["var_garch_cf"] for r in rows]) * 100
    var_mf    = np.array([r["var_mfgarch"]  for r in rows]) * 100
    xs        = np.arange(len(dates))

    fig = plt.figure(figsize=(18, 18))
    gs  = gridspec.GridSpec(5, 2, figure=fig,
                            height_ratios=[1.4, 1.2, 1.2, 1.2, 1.4],
                            hspace=0.48, wspace=0.30)
    fig.suptitle(
        f"MF-GARCH Conditional VaR  ·  {ticker}  ·  {CONFIDENCE*100:.0f}% conf  "
        f"·  min Pearson r = {min_r:.3f}",
        fontsize=13, fontweight="bold"
    )

    def xticks(ax, skip=True):
        step = max(1, len(xs) // 8)
        ax.set_xticks(xs[::step])
        if skip:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xticklabels([str(dates[i])[:10] for i in xs[::step]],
                               rotation=30, ha="right", fontsize=8)

    # (a) price
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(xs, prices, color="#1A1A2E", lw=1.2)
    ax.set_title("(a) Price", fontsize=9, fontweight="semibold")
    ax.set_ylabel("USD", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    xticks(ax); ax.grid(alpha=0.12)

    # (b) GARCH sigma
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(xs, sigma, color=COLOR_GARCH, lw=1.2)
    ax.set_title("(b) GARCH 1-step conditional σ", fontsize=9, fontweight="semibold")
    ax.set_ylabel("σ (%)", fontsize=9)
    xticks(ax); ax.grid(alpha=0.12)

    # (c) Δα
    ax = fig.add_subplot(gs[1, 0])
    valid_da = np.isfinite(da_arr)
    ax.plot(xs[valid_da], da_arr[valid_da], color="#bc8cff", lw=1.2)
    ax.axhline(np.nanmedian(da_arr), color="#555", lw=0.8, ls="--",
               label=f"Median = {np.nanmedian(da_arr):.3f}")
    ax.set_title("(c) Rolling MF-DFA Δα — singularity spectrum width",
                 fontsize=9, fontweight="semibold")
    ax.set_ylabel("Δα", fontsize=9)
    ax.legend(fontsize=8); xticks(ax); ax.grid(alpha=0.12)

    # (d) multiplier
    ax = fig.add_subplot(gs[1, 1])
    ax.plot(xs, mult_arr, color=COLOR_MFGARCH, lw=1.2)
    ax.axhline(1.0, color="#555", lw=0.7, ls="--", alpha=0.6)
    ax.fill_between(xs, 1.0, mult_arr, where=mult_arr > 1,
                    color=COLOR_MFGARCH, alpha=0.15)
    ax.set_title(f"(d) MF multiplier = 1 + {MF_LAMBDA}·clip(Δα_z, 0, {MF_Z_CAP})",
                 fontsize=9, fontweight="semibold")
    ax.set_ylabel("Multiplier", fontsize=9)
    xticks(ax); ax.grid(alpha=0.12)

    # (e) all VaR
    ax = fig.add_subplot(gs[2, 0])
    ax.plot(xs, var_std,   color=COLOR_STD,     lw=0.9, ls=":",  label="Standard hist")
    ax.plot(xs, var_garch, color=COLOR_GARCH,   lw=1.1, ls="--", label="GARCH")
    ax.plot(xs, var_gcf,   color=COLOR_GARCHCF, lw=1.3,          label="GARCH+CF")
    ax.plot(xs, var_mf,    color=COLOR_MFGARCH, lw=1.8,          label="MF-GARCH+CF")
    ax.set_title("(e) Four VaR estimates", fontsize=9, fontweight="semibold")
    ax.set_ylabel("VaR (%)", fontsize=9)
    ax.legend(fontsize=8, loc="upper right")
    xticks(ax); ax.grid(alpha=0.12)

    # (f) actual vs VaR
    ax = fig.add_subplot(gs[2, 1])
    breach_mf  = np.array([r["breach_mfgarch"] for r in rows])
    breach_std = np.array([r["breach_std"]     for r in rows])
    ax.plot(xs, actual,  color="#cccccc", lw=0.6, alpha=0.8, label="Actual loss")
    ax.plot(xs, var_mf,  color=COLOR_MFGARCH, lw=1.6, label="MF-GARCH+CF VaR")
    ax.plot(xs, var_std, color=COLOR_STD, lw=0.9, ls=":", alpha=0.7, label="Standard VaR")
    improved = breach_std & ~breach_mf
    if improved.any():
        ax.scatter(xs[improved], actual[improved], s=35, color="orange",
                   zorder=5, marker="^", label=f"Fixed by MF-GARCH ({improved.sum()})")
    if breach_mf.any():
        ax.scatter(xs[breach_mf], actual[breach_mf], s=35, color="red",
                   zorder=6, marker="x", label=f"MF-GARCH breach ({breach_mf.sum()})")
    ax.set_title("(f) Forward validation: actual loss vs VaR",
                 fontsize=9, fontweight="semibold")
    ax.set_ylabel("Loss / VaR (%)", fontsize=9)
    ax.legend(fontsize=7, loc="upper right")
    xticks(ax); ax.grid(alpha=0.12)

    # scatter panels (g–j)
    configs = [
        ("Standard",     COLOR_STD,     var_std,   "g"),
        ("GARCH",        COLOR_GARCH,   var_garch,  "h"),
        ("GARCH+CF",     COLOR_GARCHCF, var_gcf,   "i"),
        ("MF-GARCH+CF",  COLOR_MFGARCH, var_mf,    "j"),
    ]
    for k, (name, color, var_arr, lbl) in enumerate(configs):
        row_gs = 3 + k // 2
        col_gs = k % 2
        ax     = fig.add_subplot(gs[row_gs, col_gs])
        mask   = np.isfinite(var_arr) & np.isfinite(actual)
        ax.scatter(var_arr[mask], actual[mask], s=5, alpha=0.2,
                   color=color, rasterized=True)
        if mask.sum() > 4:
            m, b = np.polyfit(var_arr[mask], actual[mask], 1)
            xl   = np.linspace(var_arr[mask].min(), var_arr[mask].max(), 100)
            ax.plot(xl, m*xl + b, color=color, lw=1.8)
        lim = [min(var_arr[mask].min(), actual[mask].min()),
               max(var_arr[mask].max(), actual[mask].max())]
        ax.plot(lim, lim, color="#666", lw=0.8, ls=":")
        v = val_results[name]
        ax.set_title(f"({lbl}) {name}\nR²={v['r2']:.4f}  "
                     f"Breach={v['br_pct']:.2f}%  Sev={v['sev']*100:.3f}%",
                     fontsize=9, fontweight="semibold")
        ax.set_xlabel("Predicted VaR (%)", fontsize=9)
        ax.set_ylabel("Actual loss (%)", fontsize=9)
        ax.grid(alpha=0.12)
        xticks(ax, skip=False)

    fname = f"{OUT_DIR}/{ticker.replace('=','_').replace('-','_')}_mfgarch_var.png"
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {fname}")


def plot_cross_asset(all_val):
    tickers     = list(all_val.keys())
    model_names = ["Standard", "GARCH", "GARCH+CF", "MF-GARCH+CF"]
    colors      = [COLOR_STD, COLOR_GARCH, COLOR_GARCHCF, COLOR_MFGARCH]
    x           = np.arange(len(tickers))
    w           = 0.18

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    for ax, metric, ylabel, title in zip(
            axes,
            ["br_pct", "r2"],
            ["Breach rate (%)", "R² (VaR vs actual loss)"],
            ["Breach rates — target = 1.00%", "Predictive R² — higher = better"]):
        for i, (m, c) in enumerate(zip(model_names, colors)):
            vals = [all_val[t][m][metric] for t in tickers]
            ax.bar(x + (i - 1.5) * w, vals, width=w - 0.02,
                   color=c, alpha=0.85, label=m, edgecolor="white")
        if metric == "br_pct":
            ax.axhline(1.0, color="black", lw=1.2, ls="--", label="Target 1%")
        ax.set_xticks(x); ax.set_xticklabels(tickers, fontsize=10)
        ax.set_ylabel(ylabel); ax.set_title(title, fontsize=11)
        ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.2)

    fig.suptitle("MF-GARCH+CF vs baselines — cross-asset summary", fontsize=12)
    fname = f"{OUT_DIR}/cross_asset_summary.png"
    fig.tight_layout()
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    t_total = time.perf_counter()
    all_val = {}

    for idx, (ticker, (sector, min_r)) in enumerate(ASSETS.items()):
        t0 = time.perf_counter()
        print(f"\n{'─'*60}")
        print(f"  {ticker}  [{sector}]  min Pearson r = {min_r:.4f}")
        print(f"{'─'*60}")

        data  = yf.download(ticker, start=START, end=END, progress=False)
        close = data["Close"].dropna().values.ravel().astype(float)
        dates = data["Close"].dropna().index.tolist()
        print(f"  {len(close)} bars downloaded")

        rows    = run_asset(ticker, close, dates)
        val     = validation_summary(rows, ticker, min_r)
        all_val[ticker] = val
        plot_asset(ticker, rows, min_r, val)
        print(f"  Asset total: {time.perf_counter()-t0:.1f}s")

    plot_cross_asset(all_val)

    # ── Final summary table ───────────────────────────────────────────────
    print(f"\n{'═'*75}")
    print(f"  {'Ticker':<8} {'Model':<15} {'Breach%':>7} {'R²':>7} "
          f"{'Sev%':>8} {'Kupiec-p':>9}")
    print(f"{'─'*75}")
    for ticker, val in all_val.items():
        for model, v in val.items():
            print(f"  {ticker:<8} {model:<15} {v['br_pct']:>6.2f}% "
                  f"{v['r2']:>7.4f} {v['sev']*100:>7.3f}% "
                  f"{v['kupiec_p']:>9.4f}")
        print(f"{'─'*75}")

    print(f"\nTotal wall time: {time.perf_counter()-t_total:.1f}s")
    print(f"Plots → {OUT_DIR}/")


────────────────────────────────────────────────────────────
  KO  [Healthcare & Consumer]  min Pearson r = 0.2383
────────────────────────────────────────────────────────────
  4092 bars downloaded
  Pre-computing Δα history (every 5 steps) …
  Δα history: n=51, mean=0.5545, std=0.1659
  Loop: 3340 steps in 32.8s (9.81 ms/step)

  Validation — KO (min r=0.2383)  n=3340
  Model           Breach%  Target      R²     Sev%  Kupiec-p
  ─────────────────────────────────────────────────────────
  Standard          2.49%   1.00%  0.0002   0.938%    0.0000
  GARCH             2.37%   1.00%  0.0003   1.158%    0.0000
  GARCH+CF          1.29%   1.00%  0.0004   1.276%    0.1099
  MF-GARCH+CF       0.99%   1.00%  0.0003   1.491%    0.9444
  Saved → mf_garch_plots/KO_mfgarch_var.png
  Asset total: 35.6s

────────────────────────────────────────────────────────────
  PEP  [Healthcare & Consumer]  min Pearson r = -0.1332
────────────────────────────────────────────────────────────
  4092 bars downl

In [8]:
"""
MF-GARCH Statistical Proof Suite — Final Clean Version
=======================================================
Pulls REAL price data from yfinance.
Runs REAL rolling backtest (no synthetic data anywhere).
Applies five statistical tests on REAL computed VaR arrays:

  1. Kupiec POF              — unconditional coverage
  2. Christoffersen          — breach independence
  3. Diebold-Mariano         — TICK LOSS (not MSE — see note below)
  4. Spearman ρ(min_r, p)    — does min-r predict model quality?
  5. Per-asset λ calibration — proposed values from min-r + Δα std

WHY TICK LOSS FOR DM (Test 3):
  MSE penalises over-estimation symmetrically. A conservative VaR model
  that pads estimates on calm days accumulates huge MSE even if it never
  misses a breach. The asymmetric tick (quantile) loss at α=1% is the
  correct scoring rule for VaR:
      L(VaR, y) = α*(y-VaR)     if y > VaR   [breach — penalise hard]
                = (1-α)*(VaR-y)  if y ≤ VaR   [no breach — tiny penalty]
  DM > 0 → MF-GARCH+CF has lower tick loss → genuinely better at the tail.
  DM < 0 → MF-GARCH+CF has higher tick loss → over-inflating VaR hurts.
"""

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from scipy.signal import savgol_filter
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, os, time

warnings.filterwarnings("ignore")

# ── CONFIG ─────────────────────────────────────────────────────────────────────
ASSETS = {
    "KO":   {"minR": 0.2383,  "da_std": 0.1659},
    "PEP":  {"minR": -0.1332, "da_std": 0.1081},
    "JNJ":  {"minR": 0.3982,  "da_std": 0.0563},
    "CVX":  {"minR": 0.5253,  "da_std": 0.0504},
    "MA":   {"minR": 0.7516,  "da_std": 0.1089},
    "CL=F": {"minR": 0.7178,  "da_std": 0.0466},
    "AAPL": {"minR": 0.9700,  "da_std": 0.1564},
    "JPM":  {"minR": 0.9842,  "da_std": 0.1541},
}

START        = "2010-01-01"
END          = "2026-04-11"
CONFIDENCE   = 0.99
ALPHA        = 1 - CONFIDENCE        # 0.01
GARCH_WINDOW = 250
MF_WINDOW    = 500
CF_WINDOW    = 120
DA_HISTORY   = 250
GARCH_REFIT  = 5
MF_REFIT     = 5
MF_LAMBDA    = 0.10
MF_Z_CAP     = 3.0
Q_VALS       = np.concatenate([np.arange(-10, 0), np.arange(1, 11)]).astype(float)
SCALE_MIN    = 16
SCALE_MAX    = 250
N_SCALES     = 20
OUT_DIR      = "proof_plots_final"
os.makedirs(OUT_DIR, exist_ok=True)

# ── PROJECTION CACHE (speeds up MF-DFA) ────────────────────────────────────────
_PROJ_CACHE = {}
def _get_proj(s):
    if s not in _PROJ_CACHE:
        t = np.arange(s, dtype=np.float64)
        _PROJ_CACHE[s] = np.column_stack([t, np.ones(s)])
    return _PROJ_CACHE[s]

# ══════════════════════════════════════════════════════════════════════════════
# LAYER 1 — MF-DFA
# ══════════════════════════════════════════════════════════════════════════════
def mfdfa_hq(x, q_vals, scale_min, scale_max, n_scales, overlap=0.5):
    """
    Multi-fractal DFA. Returns generalised Hurst exponents h(q) for each q.
    h(q) constant → monofractal (Brownian-like).
    h(q) varying  → multifractal (rich scaling structure).
    """
    y         = np.cumsum(x - x.mean())
    N         = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, 3)
    if scale_max < scale_min:
        return np.full(len(q_vals), np.nan)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]
    Fqs    = np.full((len(q_vals), len(scales)), np.nan)
    log_s  = np.log(scales.astype(float))

    for i, s in enumerate(scales):
        step   = max(1, int(s * (1 - overlap)))
        starts = np.arange(0, N - s + 1, step)
        if len(starts) < 3:
            continue
        X     = _get_proj(s)
        segs  = np.stack([y[st:st + s] for st in starts])
        C, _, _, _ = np.linalg.lstsq(X, segs.T, rcond=None)
        resid = segs - (X @ C).T
        rms   = np.sqrt(np.mean(resid ** 2, axis=1))
        rms   = rms[rms > 0]
        if len(rms) == 0:
            continue
        rms2 = rms ** 2
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms2)))
            else:
                v = np.mean(rms ** q)
                if v > 0:
                    Fqs[j, i] = v ** (1 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(log_s[mask], np.log(row[mask]), 1)[0]
    return hq


def delta_alpha(hq, q_vals):
    """
    Width of the multifractal spectrum Δα = α_max - α_min.
    Larger Δα → richer multifractal structure → more signal for MF layer.
    """
    valid = np.isfinite(hq)
    if valid.sum() < 6:
        return np.nan
    q_v   = q_vals[valid]
    tau   = q_v * hq[valid] - 1.0
    wl    = min(5, len(tau) if len(tau) % 2 == 1 else len(tau) - 1)
    tau_s = (savgol_filter(tau, window_length=max(wl, 3), polyorder=2)
             if len(tau) >= 5 else tau)
    alpha = np.gradient(tau_s, q_v)
    fin   = np.isfinite(alpha)
    return float(alpha[fin].max() - alpha[fin].min()) if fin.sum() >= 2 else np.nan


# ══════════════════════════════════════════════════════════════════════════════
# LAYER 2 — GARCH(1,1)
# ══════════════════════════════════════════════════════════════════════════════
_GARCH_BOUNDS = [(1e-8, None), (1e-6, 0.5), (1e-6, 0.9999)]


def _garch_filter(params, returns):
    omega, alpha, beta = params
    n    = len(returns)
    h    = np.empty(n)
    h[0] = max(np.var(returns), 1e-10)
    for t in range(1, n):
        h[t] = omega + alpha * returns[t - 1] ** 2 + beta * h[t - 1]
    return np.maximum(h, 1e-10)


def _garch_negll(params, returns):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return 1e10
    h = _garch_filter(params, returns)
    return 0.5 * float(np.sum(np.log(h) + returns ** 2 / h))


def garch_fit(returns, p0=None):
    if p0 is None:
        v  = np.var(returns)
        p0 = np.array([v * 0.05, 0.10, 0.85])
    res = minimize(
        _garch_negll, p0, args=(returns,), method="L-BFGS-B",
        bounds=_GARCH_BOUNDS, options={"maxiter": 200, "ftol": 1e-7}
    )
    if res.success and res.x[1] + res.x[2] < 1:
        params = res.x
    else:
        v      = np.var(returns)
        params = np.array([v * 0.01, 0.06, 0.94])
    h        = _garch_filter(params, returns)
    sigma_t1 = float(np.sqrt(
        params[0] + params[1] * returns[-1] ** 2 + params[2] * h[-1]
    ))
    return params, np.sqrt(h), sigma_t1


# ══════════════════════════════════════════════════════════════════════════════
# LAYER 3 — CORNISH-FISHER VaR CORRECTION
# ══════════════════════════════════════════════════════════════════════════════
def cf_zscore(residuals, confidence=CONFIDENCE):
    """
    Cornish-Fisher expansion adjusts the normal quantile for empirical
    skewness (g1) and excess kurtosis (g2) of GARCH residuals.
    This makes VaR sensitive to fat tails and asymmetry — crucial for
    accurate 99% coverage.
    """
    g1 = float(stats.skew(residuals))
    g2 = float(stats.kurtosis(residuals))
    z  = stats.norm.ppf(1 - confidence)
    return float(
        z + (z ** 2 - 1) * g1 / 6
        + (z ** 3 - 3 * z) * g2 / 24
        - (2 * z ** 3 - 5 * z) * g1 ** 2 / 36
    )


# ══════════════════════════════════════════════════════════════════════════════
# ROLLING BACKTEST — produces real VaR arrays from real price data
# ══════════════════════════════════════════════════════════════════════════════
def run_asset(ticker, close, dates):
    """
    Full rolling out-of-sample backtest.
    For each day i we use ONLY data up to i-1 to produce the VaR forecast.
    No look-ahead. Returns list of dicts with actual_loss and four VaR estimates.
    """
    returns   = np.diff(np.log(close[close > 0]))
    N         = len(returns)
    start_idx = max(GARCH_WINDOW, MF_WINDOW, CF_WINDOW) + DA_HISTORY

    # Pre-compute rolling Δα history (used to normalise the MF signal)
    da_raw = []
    for i in range(MF_WINDOW, min(start_idx + 1, N - 1), MF_REFIT):
        seg = returns[i - MF_WINDOW:i]
        hq  = mfdfa_hq(seg, Q_VALS, SCALE_MIN, min(SCALE_MAX, len(seg) // 4), N_SCALES)
        da_raw.append(delta_alpha(hq, Q_VALS))
    da_raw        = np.array(da_raw)
    da_med_global = np.nanmedian(da_raw)
    da_std_global = max(np.nanstd(da_raw), 1e-6)

    rows         = []
    garch_params = None
    cached_mult  = 1.0
    z_base       = stats.norm.ppf(1 - CONFIDENCE)

    for i in range(start_idx, N - 1):
        seg_g  = returns[i - GARCH_WINDOW:i]
        seg_mf = returns[i - MF_WINDOW:i]

        # GARCH — refit every GARCH_REFIT steps
        if (i - start_idx) % GARCH_REFIT == 0:
            garch_params, sigma_hist, sigma_t1 = garch_fit(seg_g, garch_params)
        else:
            sigma_hist = np.sqrt(_garch_filter(garch_params, seg_g))
            sigma_t1   = float(np.sqrt(
                garch_params[0]
                + garch_params[1] * seg_g[-1] ** 2
                + garch_params[2] * sigma_hist[-1] ** 2
            ))

        # Cornish-Fisher z-score from recent standardised residuals
        mu     = float(np.mean(seg_g))
        seg_cf = seg_g[-CF_WINDOW:]
        sig_cf = sigma_hist[-CF_WINDOW:]
        resids = (seg_cf - mu) / np.maximum(sig_cf, 1e-8)
        z_cf   = cf_zscore(resids) if len(resids) >= 20 else z_base

        # MF multiplier — recompute every MF_REFIT steps
        if (i - start_idx) % MF_REFIT == 0:
            hq     = mfdfa_hq(
                seg_mf, Q_VALS, SCALE_MIN, min(SCALE_MAX, len(seg_mf) // 4), N_SCALES
            )
            da_val = delta_alpha(hq, Q_VALS)
            h_end  = (i - start_idx) // MF_REFIT
            h_start= max(0, h_end - DA_HISTORY // MF_REFIT)
            h_slice= da_raw[h_start:h_end + 1]
            da_med = (np.nanmedian(h_slice) if np.isfinite(h_slice).any()
                      else da_med_global)
            da_std = (max(np.nanstd(h_slice), 1e-6) if np.isfinite(h_slice).any()
                      else da_std_global)
            da_z   = (np.clip((da_val - da_med) / da_std, 0, MF_Z_CAP)
                      if np.isfinite(da_val) else 0.0)
            cached_mult = 1.0 + MF_LAMBDA * da_z

        # Four VaR estimates (all positive — represent loss threshold)
        actual_loss  = -returns[i]
        var_std      = max(float(-np.percentile(seg_g[-60:], 1)), 1e-6)
        var_garch    = max(-(mu + z_base * sigma_t1), 1e-6)
        var_garch_cf = max(-(mu + z_cf   * sigma_t1), 1e-6)
        var_mfgarch  = max(var_garch_cf * cached_mult, 1e-6)

        rows.append({
            "date"            : dates[i + 1],
            "actual_loss"     : actual_loss,
            "var_std"         : var_std,
            "var_garch"       : var_garch,
            "var_garch_cf"    : var_garch_cf,
            "var_mfgarch"     : var_mfgarch,
            "breach_std"      : int(actual_loss > var_std),
            "breach_garch"    : int(actual_loss > var_garch),
            "breach_garch_cf" : int(actual_loss > var_garch_cf),
            "breach_mfgarch"  : int(actual_loss > var_mfgarch),
        })
    return rows


# ══════════════════════════════════════════════════════════════════════════════
# STATISTICAL TESTS
# ══════════════════════════════════════════════════════════════════════════════
def kupiec_pof(n_breaches, n_obs, confidence=CONFIDENCE):
    """
    Likelihood ratio test: H0 = breach rate equals 1-confidence.
    p > 0.05 → fail to reject → model is well calibrated.
    """
    p0 = 1 - confidence
    if n_breaches == 0 or n_breaches >= n_obs:
        return np.nan, np.nan
    p_hat = n_breaches / n_obs
    lr    = -2 * (
        n_breaches     * np.log(p0 / p_hat)
        + (n_obs - n_breaches) * np.log((1 - p0) / (1 - p_hat))
    )
    return float(lr), float(1 - stats.chi2.cdf(lr, df=1))


def christoffersen(breach_series):
    """
    Tests independence of breaches. If p < 0.05 breaches arrive in clusters
    (volatility regime not captured by model).
    """
    b   = np.array(breach_series, dtype=int)
    n00 = np.sum((b[:-1] == 0) & (b[1:] == 0))
    n01 = np.sum((b[:-1] == 0) & (b[1:] == 1))
    n10 = np.sum((b[:-1] == 1) & (b[1:] == 0))
    n11 = np.sum((b[:-1] == 1) & (b[1:] == 1))
    if (n01 + n11) == 0 or (n00 + n10) == 0:
        return np.nan, np.nan
    p01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    p11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0
    p_  = (n01 + n11) / len(b)
    def sl(x): return np.log(x) if x > 0 else 0
    L_i = (n00 + n10) * sl(1 - p_) + (n01 + n11) * sl(p_)
    L_d = (n00 * sl(1 - p01) + n01 * sl(p01)
           + n10 * sl(1 - p11) + n11 * sl(p11))
    lr  = -2 * (L_i - L_d)
    return float(lr), float(1 - stats.chi2.cdf(lr, df=1))


def tick_loss_series(var_arr, actual, alpha=ALPHA):
    """
    Asymmetric quantile (tick) loss — the correct scoring rule for VaR.
    On breach days (actual > VaR): loss = alpha * (actual - VaR)
    On non-breach days:            loss = (1-alpha) * (VaR - actual)
    Expected value of this loss is minimised by the true conditional quantile.
    """
    e = actual - var_arr
    return np.where(e > 0, alpha * e, (1 - alpha) * (-e))


def diebold_mariano_tick(var_a, var_b, actual, alpha=ALPHA):
    """
    DM test using asymmetric tick loss with Newey-West HAC variance.

    var_a = baseline model (GARCH+CF)
    var_b = challenger model (MF-GARCH+CF)

    d_t = L(var_a, actual_t) - L(var_b, actual_t)
    DM > 0 → challenger has lower loss on average → MF is better at the tail
    DM < 0 → challenger has higher loss           → MF is worse at the tail

    HAC (Newey-West) used because VaR forecast errors are autocorrelated.
    Lag truncation = floor(T^(1/3)) per standard practice.
    """
    L_a   = tick_loss_series(var_a, actual, alpha)
    L_b   = tick_loss_series(var_b, actual, alpha)
    d     = L_a - L_b
    T     = len(d)
    d_bar = d.mean()

    # Newey-West HAC variance
    lags  = max(1, int(np.floor(T ** (1 / 3))))
    gamma0= np.var(d, ddof=1)
    nw_var= gamma0
    for l in range(1, lags + 1):
        gl     = np.cov(d[l:], d[:T - l], ddof=1)[0, 1]
        nw_var += 2 * (1 - l / (lags + 1)) * gl
    var_d = max(nw_var / T, 1e-20)

    dm   = d_bar / np.sqrt(var_d)
    pval = 2 * (1 - stats.norm.cdf(abs(dm)))
    return float(dm), float(pval), float(L_a.mean()), float(L_b.mean())


def required_n_spearman(rho, power=0.80, alpha_level=0.05):
    """Minimum sample size for Spearman test at given power."""
    if not np.isfinite(rho) or abs(rho) < 1e-6:
        return np.inf
    z_alpha = stats.norm.ppf(1 - alpha_level / 2)
    z_beta  = stats.norm.ppf(power)
    z_rho   = np.arctanh(abs(rho))
    return int(np.ceil((z_alpha + z_beta) ** 2 / z_rho ** 2 + 3))


# ══════════════════════════════════════════════════════════════════════════════
# PER-ASSET REASONING — plain-English interpretation of all test evidence
# ══════════════════════════════════════════════════════════════════════════════
def print_asset_reasoning(ticker, n, nb_cf, nb_mf,
                          kup_p_cf, kup_p_mf,
                          chr_p_mf,
                          dm, dm_p, L_cf, L_mf,
                          min_r, da_std_v):
    target = n * ALPHA
    delta_kup = kup_p_mf - kup_p_cf
    pct_loss_change = (L_cf - L_mf) / max(L_cf, 1e-15) * 100

    print(f"\n  ┌─ {ticker} ─────────────────────────────────────────────────────────")
    print(f"  │  n={n}  target breaches≈{target:.0f}  "
          f"GARCH+CF={nb_cf} ({nb_cf/n*100:.2f}%)  "
          f"MF-GARCH+CF={nb_mf} ({nb_mf/n*100:.2f}%)")

    # ── Kupiec
    if kup_p_mf > 0.05 and kup_p_cf <= 0.05:
        print(f"  │  KUPIEC : MF fixes calibration. CF fails (p={kup_p_cf:.4f}) "
              f"→ MF passes (p={kup_p_mf:.4f}).")
        print(f"  │           Excess breaches: CF={nb_cf-int(target):+d} → MF={nb_mf-int(target):+d}.")
    elif kup_p_mf > 0.05 and kup_p_cf > 0.05:
        print(f"  │  KUPIEC : Both pass. MF shifts p by {delta_kup:+.4f} "
              f"({'away from' if delta_kup > 0 else 'toward'} rejection).")
    elif kup_p_mf <= 0.05 and kup_p_cf <= 0.05:
        print(f"  │  KUPIEC : Both FAIL. MF {'improves' if delta_kup>0 else 'worsens'} "
              f"p by {delta_kup:+.4f} — still over-breaching.")
    else:
        print(f"  │  KUPIEC : CF passes (p={kup_p_cf:.4f}) but MF fails (p={kup_p_mf:.4f}). "
              f"MF over-inflates VaR.")

    # ── Christoffersen
    if np.isfinite(chr_p_mf):
        if chr_p_mf > 0.05:
            print(f"  │  CHRISTO: No breach clustering (p={chr_p_mf:.4f}). "
                  f"Temporal structure captured.")
        else:
            print(f"  │  CHRISTO: Breach CLUSTERING (p={chr_p_mf:.4f}). "
                  f"Breaches arrive in runs — volatility regime not fully captured.")

    # ── DM tick loss
    if dm > 0 and dm_p < 0.05:
        print(f"  │  DM-TICK: MF significantly BETTER at tail (DM={dm:+.4f}, p={dm_p:.4f}).")
        print(f"  │           Tick loss: CF={L_cf:.6f} → MF={L_mf:.6f} "
              f"({abs(pct_loss_change):.2f}% reduction).")
    elif dm < 0 and dm_p < 0.05:
        print(f"  │  DM-TICK: MF significantly WORSE at tail (DM={dm:+.4f}, p={dm_p:.4f}).")
        print(f"  │           Tick loss: CF={L_cf:.6f} → MF={L_mf:.6f} "
              f"({abs(pct_loss_change):.2f}% increase). "
              f"MF over-inflates VaR → excess over-estimation penalty.")
    else:
        print(f"  │  DM-TICK: No significant tail-loss difference "
              f"(DM={dm:+.4f}, p={dm_p:.4f}). "
              f"CF={L_cf:.6f}, MF={L_mf:.6f}.")

    # ── Structure context
    if min_r < 0.5:
        structure = (f"min_r={min_r:.4f} (low) → strong anti-persistence / mean-reversion. "
                     f"MF signal distinct from Brownian baseline — Δα enrichment meaningful.")
    elif min_r < 0.75:
        structure = (f"min_r={min_r:.4f} (moderate) → mild persistence. "
                     f"MF adds some structure but signal weakening.")
    else:
        structure = (f"min_r={min_r:.4f} (high, near BM) → near-random-walk. "
                     f"Small Δα spread relative to noise — MF multiplier adds little information.")
    print(f"  │  STRUCT : {structure}")
    print(f"  │           Δα temporal std={da_std_v:.4f} "
          f"({'high variability → active MF scaling' if da_std_v > 0.10 else 'low variability → stable MF scaling'}).")

    print(f"  └────────────────────────────────────────────────────────────────────")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    t_total     = time.perf_counter()
    all_rows    = {}
    model_names = ["Standard", "GARCH", "GARCH+CF", "MF-GARCH+CF"]

    # ── Data download + backtest ───────────────────────────────────────────────
    for ticker in ASSETS:
        t0   = time.perf_counter()
        print(f"\n── {ticker} ──")
        data  = yf.download(ticker, start=START, end=END, progress=False)
        close = data["Close"].dropna().values.ravel().astype(float)
        dates = data["Close"].dropna().index.tolist()
        print(f"  {len(close)} bars downloaded")
        if len(close) < MF_WINDOW + DA_HISTORY + 10:
            print(f"  !! Insufficient data — skipping {ticker}")
            continue
        rows = run_asset(ticker, close, dates)
        all_rows[ticker] = rows
        print(f"  {len(rows)} backtest steps  ({time.perf_counter()-t0:.1f}s)")

    tickers = [t for t in ASSETS if t in all_rows]

    # ── TEST 1 & 2: Kupiec + Christoffersen ───────────────────────────────────
    print(f"\n{'═'*80}")
    print("  TEST 1 — Kupiec POF  |  TEST 2 — Christoffersen")
    print(f"{'─'*80}")
    print(f"  {'Ticker':<6} {'Model':<14} {'N':>5} {'Breach':>7} {'Br%':>6}"
          f"  {'Kup-LR':>7} {'Kup-p':>7} {'Chr-p':>7} {'Pass':>5}")
    print(f"  {'─'*75}")

    test_results = {}
    for ticker in tickers:
        rows    = all_rows[ticker]
        actual  = np.array([r["actual_loss"]    for r in rows])
        var_std = np.array([r["var_std"]         for r in rows])
        var_g   = np.array([r["var_garch"]       for r in rows])
        var_gcf = np.array([r["var_garch_cf"]    for r in rows])
        var_mf  = np.array([r["var_mfgarch"]     for r in rows])
        b_std   = np.array([r["breach_std"]          for r in rows], dtype=int)
        b_g     = np.array([r["breach_garch"]        for r in rows], dtype=int)
        b_gcf   = np.array([r["breach_garch_cf"]     for r in rows], dtype=int)
        b_mf    = np.array([r["breach_mfgarch"]      for r in rows], dtype=int)

        n = len(rows)
        var_arrays    = [var_std, var_g, var_gcf, var_mf]
        breach_arrays = [b_std,   b_g,   b_gcf,   b_mf]
        test_results[ticker] = {}

        for mname, var_arr, breach_arr in zip(model_names, var_arrays, breach_arrays):
            nb           = int(breach_arr.sum())
            br_pct       = nb / n * 100
            kup_lr, kup_p= kupiec_pof(nb, n)
            _,      chr_p= christoffersen(breach_arr)
            kup_p = kup_p if kup_p is not None else np.nan
            chr_p = chr_p if chr_p is not None else np.nan
            passed = ("✓" if kup_p > 0.05 and chr_p > 0.05 else
                      "½" if kup_p > 0.05 else "✗")
            test_results[ticker][mname] = {
                "n": n, "nb": nb, "br_pct": br_pct,
                "kup_lr": kup_lr, "kup_p": kup_p, "chr_p": chr_p,
                "var_arr": var_arr, "breach_arr": breach_arr, "actual": actual,
            }
            lr_str = f"{kup_lr:.3f}" if np.isfinite(kup_lr) else "   nan"
            kp_str = f"{kup_p:.4f}"  if np.isfinite(kup_p)  else "  nan"
            cp_str = f"{chr_p:.4f}"  if np.isfinite(chr_p)  else "  nan"
            print(f"  {ticker:<6} {mname:<14} {n:>5} {nb:>7} {br_pct:>6.2f}%"
                  f"  {lr_str:>7}  {kp_str:>7}  {cp_str:>7}  {passed:>5}")
        print()

    # ── TEST 3: Diebold-Mariano with TICK LOSS ─────────────────────────────────
    print(f"\n{'═'*80}")
    print("  TEST 3 — Diebold-Mariano: MF-GARCH+CF vs GARCH+CF")
    print("  Loss = asymmetric TICK LOSS at α=1% (NOT squared error)")
    print("  DM > 0 + p < 0.05  →  MF genuinely better at the 1% tail")
    print("  DM < 0 + p < 0.05  →  MF over-inflates VaR, higher tail loss")
    print(f"{'─'*80}")
    print(f"  {'Ticker':<6} {'DM stat':>9} {'p-val':>8}"
          f"  {'L_CF':>10} {'L_MF':>10} {'Δloss%':>8}  Verdict")
    print(f"  {'─'*70}")

    dm_results = {}
    for ticker in tickers:
        r_gcf  = test_results[ticker]["GARCH+CF"]
        r_mf   = test_results[ticker]["MF-GARCH+CF"]
        actual = r_mf["actual"]
        dm, p, L_cf, L_mf = diebold_mariano_tick(
            r_gcf["var_arr"], r_mf["var_arr"], actual
        )
        delta_l = (L_cf - L_mf) / max(L_cf, 1e-15) * 100
        verdict = ("MF better at tail ✓"     if dm > 0 and p < 0.05 else
                   "MF worse at tail  ✗"     if dm < 0 and p < 0.05 else
                   "no significant difference")
        dm_results[ticker] = {
            "dm": dm, "p": p, "L_cf": L_cf, "L_mf": L_mf,
            "delta_l": delta_l, "verdict": verdict,
        }
        print(f"  {ticker:<6} {dm:>+9.4f} {p:>8.4f}"
              f"  {L_cf:>10.6f} {L_mf:>10.6f} {delta_l:>+8.2f}%  {verdict}")

    print()
    print("  Interpretation: tick loss = α*breach_shortfall + (1-α)*over_estimation.")
    print("  A model that pads VaR on every calm day still pays 0.99*(excess) per day.")
    print("  DM < 0 means the over-estimation cost outweighs the breach savings.")

    # ── TEST 4: Spearman ρ ─────────────────────────────────────────────────────
    print(f"\n{'═'*80}")
    print("  TEST 4 — Spearman ρ: does min-r predict where MF helps?")
    print(f"{'─'*80}")

    min_rs   = np.array([ASSETS[t]["minR"]   for t in tickers])
    da_stds  = np.array([ASSETS[t]["da_std"] for t in tickers])
    kp_mf    = np.array([test_results[t]["MF-GARCH+CF"]["kup_p"] for t in tickers])
    kp_cf    = np.array([test_results[t]["GARCH+CF"]["kup_p"]    for t in tickers])
    dm_stats = np.array([dm_results[t]["dm"]                     for t in tickers])

    n_assets = len(tickers)
    t_crit   = stats.t.ppf(0.975, df=max(n_assets - 2, 1))
    rho_crit = t_crit / np.sqrt(t_crit ** 2 + n_assets - 2)

    print(f"  n={n_assets} assets. Need |ρ| > {rho_crit:.3f} for significance at α=0.05.")
    print()

    pairs = [
        ("min_r  → Kupiec-p(MF)",       min_rs,  kp_mf),
        ("da_std → Kupiec-p(MF)",        da_stds, kp_mf),
        ("min_r  → Kupiec-p(GARCH+CF)",  min_rs,  kp_cf),
        ("min_r  → DM-tick statistic",   min_rs,  dm_stats),
    ]
    spearman_results = {}
    for label, x, y in pairs:
        rho, p = stats.spearmanr(x, y)
        n_req  = required_n_spearman(rho)
        sig    = "★ SIGNIFICANT" if p < 0.05 else "not significant"
        power  = ("sufficient" if n_assets >= n_req else
                  f"UNDERPOWERED — need n≥{n_req} for 80% power")
        spearman_results[label] = {"rho": rho, "p": p}
        print(f"  ρ({label}) = {rho:+.4f}  p={p:.4f}  {sig}")
        print(f"    → {power}")
        # Plain-English interpretation
        if "Kupiec-p(MF)" in label and "min_r" in label:
            if rho > 0:
                print(f"    → Higher min_r (near-BM) associates with higher Kupiec-p(MF).")
                print(f"       Directionally: MF calibrates well on near-BM assets too,")
                print(f"       but the relationship is {'confirmed' if p<0.05 else 'unconfirmed at this sample size'}.")
            else:
                print(f"    → Higher min_r (near-BM) associates with LOWER Kupiec-p(MF).")
                print(f"       MF may over-fit on near-random-walk assets.")
        if "DM-tick" in label:
            if rho > 0:
                print(f"    → Higher min_r associates with higher DM stat (MF more competitive).")
            else:
                print(f"    → Higher min_r associates with lower DM stat (MF less competitive).")
        print()

    # ── TEST 5: Per-asset λ calibration ────────────────────────────────────────
    print(f"{'═'*80}")
    print("  TEST 5 — Per-asset λ calibration")
    print()
    print("  Formula:")
    print("    richness  = max(0, 1 - min_r)")
    print("    da_qual   = da_std / median(da_stds)")
    print("    scale     = clip(richness/median(richness) * da_qual, 0.1, 2.0)")
    print("    λ_proposed = λ_global * scale")
    print()
    print("  Rationale:")
    print("    min_r near 0   → strong anti-persistence → large Δα spread → rich MF signal → boost λ")
    print("    min_r near 1   → near-Brownian           → weak Δα signal  → reduce λ")
    print("    high da_std    → MF spectrum varies dynamically → scaling is responsive → keep/boost λ")
    print(f"{'─'*80}")
    print(f"  {'Ticker':<6} {'min_r':>7} {'richness':>9} {'da_qual':>8}"
          f"  {'λ_curr':>8} {'λ_prop':>8}  direction  rationale")
    print(f"  {'─'*75}")

    med_rich = max(np.median([max(0, 1 - ASSETS[t]["minR"]) for t in tickers]), 0.01)
    med_ds   = np.median([ASSETS[t]["da_std"] for t in tickers])
    proposed_lambdas = {}

    for ticker in tickers:
        mr_   = ASSETS[ticker]["minR"]
        ds_   = ASSETS[ticker]["da_std"]
        rich  = max(0.0, 1.0 - mr_)
        daqual= ds_ / med_ds
        scale = np.clip((rich / med_rich) * daqual, 0.1, 2.0)
        lam   = round(MF_LAMBDA * scale, 4)
        proposed_lambdas[ticker] = lam
        dirn  = ("↑ boost"  if lam > MF_LAMBDA * 1.1 else
                 "↓ reduce" if lam < MF_LAMBDA * 0.5  else "→ keep")
        why   = ("strong MF signal, high variability"   if rich > 0.4 and ds_ > med_ds else
                 "strong MF signal, stable spectrum"    if rich > 0.4 else
                 "weak MF signal, high variability"     if rich <= 0.4 and ds_ > med_ds else
                 "near-BM, low Δα variability → reduce")
        print(f"  {ticker:<6} {mr_:>7.4f} {rich:>9.4f} {daqual:>8.4f}"
              f"  {MF_LAMBDA:>8.4f} {lam:>8.4f}  {dirn:<10} {why}")

    # ── PER-ASSET REASONING ────────────────────────────────────────────────────
    print(f"\n{'═'*80}")
    print("  PER-ASSET REASONING — full synthesis of all five tests")
    print(f"{'═'*80}")
    for ticker in tickers:
        r_cf = test_results[ticker]["GARCH+CF"]
        r_mf = test_results[ticker]["MF-GARCH+CF"]
        dmr  = dm_results[ticker]
        print_asset_reasoning(
            ticker     = ticker,
            n          = r_mf["n"],
            nb_cf      = r_cf["nb"],
            nb_mf      = r_mf["nb"],
            kup_p_cf   = r_cf["kup_p"],
            kup_p_mf   = r_mf["kup_p"],
            chr_p_mf   = r_mf["chr_p"],
            dm         = dmr["dm"],
            dm_p       = dmr["p"],
            L_cf       = dmr["L_cf"],
            L_mf       = dmr["L_mf"],
            min_r      = ASSETS[ticker]["minR"],
            da_std_v   = ASSETS[ticker]["da_std"],
        )

    # ── FINAL VERDICT ──────────────────────────────────────────────────────────
    print(f"\n{'═'*80}")
    print("  FINAL VERDICT")
    print("  Logic: (1) does MF fix/improve Kupiec?  (2) does tick-loss DM confirm?")
    print(f"{'─'*80}")
    print(f"  {'Ticker':<6} {'CF-p':>7} {'MF-p':>7} {'ΔKup':>8}"
          f"  {'DM':>8} {'DM-p':>7}  Verdict")
    print(f"  {'─'*72}")

    for ticker in tickers:
        kp_cf_v = test_results[ticker]["GARCH+CF"]["kup_p"]
        kp_mf_v = test_results[ticker]["MF-GARCH+CF"]["kup_p"]
        dkup    = kp_mf_v - kp_cf_v
        dm_v    = dm_results[ticker]["dm"]
        dm_p_v  = dm_results[ticker]["p"]

        if dkup > 0.05 and dm_v > 0 and dm_p_v < 0.05:
            verdict = "★ MF STRONGLY USEFUL  — calibration + tail accuracy confirmed"
        elif dkup > 0.05:
            verdict = "✓ MF USEFUL           — calibration improved (tail accuracy neutral)"
        elif dkup < -0.05 and kp_mf_v < 0.05:
            verdict = "✗ MF HARMFUL          — degrades calibration"
        elif dm_v < 0 and dm_p_v < 0.05:
            verdict = "~ MF COSTLY           — neutral calibration but tail loss increases"
        else:
            verdict = "~ MF NEUTRAL          — no meaningful gain or loss"

        print(f"  {ticker:<6} {kp_cf_v:>7.4f} {kp_mf_v:>7.4f} {dkup:>+8.4f}"
              f"  {dm_v:>+8.4f} {dm_p_v:>7.4f}  {verdict}")

    print(f"\n  Proposed λ values:")
    for ticker in tickers:
        print(f"  {ticker:<6}  λ_current={MF_LAMBDA:.4f}  λ_proposed={proposed_lambdas[ticker]:.4f}")

    print(f"\nTotal elapsed: {time.perf_counter()-t_total:.1f}s")

    # ── PLOTS ──────────────────────────────────────────────────────────────────
    colors = {
        "Standard"   : "#888780",
        "GARCH"      : "#58a6ff",
        "GARCH+CF"   : "#e3b341",
        "MF-GARCH+CF": "#f85149",
    }
    fig = plt.figure(figsize=(22, 20))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.52, wspace=0.38)
    fig.suptitle(
        "MF-GARCH+CF Proof Suite — Real yfinance Data, Tick-Loss DM",
        fontsize=14, fontweight="bold",
    )

    # A — breach rates
    ax = fig.add_subplot(gs[0, :2])
    x  = np.arange(len(tickers))
    w  = 0.18
    for i, m in enumerate(model_names):
        vals = [test_results[t][m]["br_pct"] for t in tickers]
        ax.bar(x + (i - 1.5) * w, vals, width=w - 0.02,
               color=colors[m], alpha=0.85, label=m, edgecolor="white")
    ax.axhline(1.0, color="black", lw=1.2, ls="--", label="Target 1%")
    ax.set_xticks(x); ax.set_xticklabels(tickers, fontsize=10)
    ax.set_ylabel("Breach rate (%)"); ax.set_title("(A) Breach rates — target = 1%")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.2)

    # B — Kupiec p-values
    ax = fig.add_subplot(gs[0, 2])
    for m, c in colors.items():
        pvals = [test_results[t][m]["kup_p"] for t in tickers]
        ax.plot(range(len(tickers)), pvals, "o-", color=c, label=m, lw=1.5, markersize=6)
    ax.axhline(0.05, color="black", lw=1, ls="--", alpha=0.6, label="α=0.05")
    ax.set_xticks(range(len(tickers))); ax.set_xticklabels(tickers, fontsize=9, rotation=30)
    ax.set_ylabel("Kupiec p-value"); ax.set_title("(B) Kupiec p-values")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

    # C — Christoffersen p-values
    ax = fig.add_subplot(gs[1, 0])
    for m, c in colors.items():
        pvals = [test_results[t][m]["chr_p"] for t in tickers]
        ax.plot(range(len(tickers)), pvals, "s--", color=c, label=m, lw=1.2, markersize=5)
    ax.axhline(0.05, color="black", lw=1, ls="--", alpha=0.6)
    ax.set_xticks(range(len(tickers))); ax.set_xticklabels(tickers, fontsize=9, rotation=30)
    ax.set_ylabel("Christoffersen p"); ax.set_title("(C) Independence test\n(p>0.05 = no clustering)")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

    # D — DM tick-loss bars
    ax = fig.add_subplot(gs[1, 1])
    dm_vals = [dm_results[t]["dm"] for t in tickers]
    dm_ps   = [dm_results[t]["p"]  for t in tickers]
    bcolors = [("#3fb950" if d > 0 and p < 0.05 else
                "#f85149" if d < 0 and p < 0.05 else "#888780")
               for d, p in zip(dm_vals, dm_ps)]
    ax.bar(range(len(tickers)), dm_vals, color=bcolors, alpha=0.85, edgecolor="white")
    ax.axhline(0, color="black", lw=0.8)
    for i, (d, p) in enumerate(zip(dm_vals, dm_ps)):
        if p < 0.05:
            ax.text(i, d + (0.04 if d > 0 else -0.15), "★", ha="center", fontsize=13)
    ax.set_xticks(range(len(tickers))); ax.set_xticklabels(tickers, fontsize=9, rotation=30)
    ax.set_ylabel("DM statistic (tick loss)")
    ax.set_title("(D) DM Test — Tick Loss\ngreen=MF better  ★=p<0.05")
    ax.grid(axis="y", alpha=0.2)

    # E — tick loss levels
    ax = fig.add_subplot(gs[1, 2])
    L_cfs = [dm_results[t]["L_cf"] for t in tickers]
    L_mfs = [dm_results[t]["L_mf"] for t in tickers]
    xi    = np.arange(len(tickers))
    ax.plot(xi, L_cfs, "o-", color="#e3b341", lw=1.5, markersize=7, label="GARCH+CF")
    ax.plot(xi, L_mfs, "s-", color="#f85149", lw=1.5, markersize=7, label="MF-GARCH+CF")
    for i, (lc, lm) in enumerate(zip(L_cfs, L_mfs)):
        color = "#3fb950" if lm < lc else "#f85149"
        ax.annotate("", xy=(i, lm), xytext=(i, lc),
                    arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
    ax.set_xticks(xi); ax.set_xticklabels(tickers, fontsize=9, rotation=30)
    ax.set_ylabel("Mean tick loss (α=1%)")
    ax.set_title("(E) Tick loss levels\narrow = direction of change")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)

    # F — Kupiec-p gain from MF
    ax = fig.add_subplot(gs[2, 0])
    dkups = [test_results[t]["MF-GARCH+CF"]["kup_p"] - test_results[t]["GARCH+CF"]["kup_p"]
             for t in tickers]
    bcs   = ["#3fb950" if d > 0 else "#f85149" for d in dkups]
    ax.bar(range(len(tickers)), dkups, color=bcs, alpha=0.85, edgecolor="white")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(range(len(tickers))); ax.set_xticklabels(tickers, fontsize=9, rotation=30)
    ax.set_ylabel("Δ Kupiec-p (MF − CF)")
    ax.set_title("(F) Kupiec-p gain from MF\ngreen=improved")
    ax.grid(axis="y", alpha=0.2)

    # G — proposed lambdas
    ax = fig.add_subplot(gs[2, 1])
    lv  = [proposed_lambdas[t] for t in tickers]
    lc2 = ["#3fb950" if l > MF_LAMBDA else "#e3b341" for l in lv]
    ax.bar(range(len(tickers)), lv, color=lc2, alpha=0.85, edgecolor="white")
    ax.axhline(MF_LAMBDA, color="#f85149", lw=1.5, ls="--", label=f"Current λ={MF_LAMBDA}")
    ax.set_xticks(range(len(tickers))); ax.set_xticklabels(tickers, fontsize=9, rotation=30)
    ax.set_ylabel("λ"); ax.set_title("(G) Per-asset λ calibration")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.2)

    # H — min-r vs Kupiec-p scatter with Spearman annotation
    ax  = fig.add_subplot(gs[2, 2])
    sc  = ax.scatter(min_rs, kp_mf, c=da_stds, cmap="RdYlGn", s=160,
                     vmin=0.04, vmax=0.18, zorder=3, edgecolors="#333", lw=0.5)
    plt.colorbar(sc, ax=ax, label="Δα temporal std")
    for t, xv, yv in zip(tickers, min_rs, kp_mf):
        ax.annotate(t, (xv, yv), textcoords="offset points", xytext=(5, 4), fontsize=9)
    if len(min_rs) > 2:
        m_, b_ = np.polyfit(min_rs, kp_mf, 1)
        xl = np.linspace(min_rs.min(), min_rs.max(), 100)
        ax.plot(xl, m_ * xl + b_, color="#888", lw=1.2, ls="--")
    ax.axhline(0.05, color="black", lw=0.8, ls=":", alpha=0.5)
    rho_mr, p_mr = spearman_results["min_r  → Kupiec-p(MF)"]["rho"], spearman_results["min_r  → Kupiec-p(MF)"]["p"]
    n_req_str = str(required_n_spearman(rho_mr)) if np.isfinite(required_n_spearman(rho_mr)) else "∞"
    ax.set_xlabel("min-r"); ax.set_ylabel("Kupiec-p (MF-GARCH+CF)")
    ax.set_title(f"(H) min-r vs Kupiec-p\nρ={rho_mr:.3f} p={p_mr:.3f}  (need n≥{n_req_str})")
    ax.grid(alpha=0.2)

    fname = f"{OUT_DIR}/mfgarch_proof_final.png"
    fig.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\nPlot saved → {fname}")


── KO ──
  4092 bars downloaded
  3340 backtest steps  (11.7s)

── PEP ──
  4092 bars downloaded
  3340 backtest steps  (42.9s)

── JNJ ──
  4092 bars downloaded
  3340 backtest steps  (29.5s)

── CVX ──
  4092 bars downloaded
  3340 backtest steps  (8.7s)

── MA ──
  4092 bars downloaded
  3340 backtest steps  (9.9s)

── CL=F ──
  4092 bars downloaded
  3339 backtest steps  (8.9s)

── AAPL ──
  4092 bars downloaded
  3340 backtest steps  (8.2s)

── JPM ──
  4092 bars downloaded
  3340 backtest steps  (10.1s)

════════════════════════════════════════════════════════════════════════════════
  TEST 1 — Kupiec POF  |  TEST 2 — Christoffersen
────────────────────────────────────────────────────────────────────────────────
  Ticker Model              N  Breach    Br%   Kup-LR   Kup-p   Chr-p  Pass
  ───────────────────────────────────────────────────────────────────────────
  KO     Standard        3340      83   2.49%   52.655   0.0000   0.2180      ✗
  KO     GARCH           3340      79